In [10]:

# LOAD AND VALIDATE THE MODELLING DATASET


# Objectives:
# 1. Import libraries and locate the project.
# 2. Load the cleaned binary modelling dataset.
# 3. Validate the population and outcome labels.
# 4. Inspect column types and missing values.
# 5. Check appointment identifiers and chronology.
# 6. Reconstruct booking_notice_group if missing.
# 7. Validate the seven-day prediction cutoff.
# 8. Identify leakage-sensitive columns.
# 9. Save initial feature engineering audit reports.
#
# No rows are removed, no values are imputed,
# and no models are trained in this cell.


# 1. Import the required libraries.

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

print("CELL 1: LOAD AND VALIDATE MODELLING DATASET")

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")


# 2. Locate the project root.

current_directory = Path.cwd()

if (
    current_directory
    / "data"
    / "processed"
).is_dir():

    PROJECT_ROOT = current_directory

elif (
    current_directory.parent
    / "data"
    / "processed"
).is_dir():

    PROJECT_ROOT = current_directory.parent

else:

    raise FileNotFoundError(
        "Project root not found. Open the "
        "nhs-dorset-dna-prediction project in VS Code."
    )

print(f"\nProject root: {PROJECT_ROOT}")


# 3. Define input and output paths.

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "outpatient_dna_modelling.csv"
)

FEATURE_REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "feature_engineering"
)

FEATURE_REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

assert DATA_PATH.is_file(), (
    f"Modelling dataset not found: {DATA_PATH}"
)

print(f"Input dataset: {DATA_PATH}")
print(f"Report directory: {FEATURE_REPORT_DIR}")


# 4. Load the modelling dataset.

print("\n1. LOAD MODELLING DATASET")

df = pd.read_csv(
    DATA_PATH,
    low_memory=False
)

# Keep a separate copy of the original data.

df_original = df.copy(deep=True)

# Work on a feature engineering copy.

df_fe = df.copy(deep=True)

print(f"Rows loaded: {len(df_fe):,}")
print(f"Original columns: {df_fe.shape[1]}")

display(df_fe.head())


# 5. Validate the modelling population.

print("\n2. MODELLING POPULATION VALIDATION")

expected_rows = 36264
expected_dnas = 3306
expected_attended = 32958

assert "dna_target" in df_fe.columns

assert df_fe["dna_target"].notna().all(), (
    "Missing target values detected."
)

assert df_fe["dna_target"].isin([0, 1]).all(), (
    "Target must contain only 0 and 1."
)

actual_rows = len(df_fe)

actual_dnas = int(
    df_fe["dna_target"].sum()
)

actual_attended = actual_rows - actual_dnas

assert actual_rows == expected_rows, (
    f"Expected {expected_rows:,} rows, "
    f"found {actual_rows:,}."
)

assert actual_dnas == expected_dnas, (
    f"Expected {expected_dnas:,} DNAs, "
    f"found {actual_dnas:,}."
)

assert actual_attended == expected_attended, (
    f"Expected {expected_attended:,} attended, "
    f"found {actual_attended:,}."
)

dna_rate = (
    actual_dnas / actual_rows * 100
)

population_summary = pd.DataFrame([
    {
        "outcome": "Attended",
        "appointments": actual_attended,
        "percentage": round(
            actual_attended / actual_rows * 100,
            2
        )
    },
    {
        "outcome": "DNA",
        "appointments": actual_dnas,
        "percentage": round(dna_rate, 2)
    }
])

display(population_summary)

print(f"Overall DNA rate: {dna_rate:.2f}%")
print("Population validation: PASS")


# 6. Parse required datetime columns.
# These dates are necessary for chronological checks.

print("\n3. DATETIME VALIDATION")

datetime_columns = [
    "booked_datetime",
    "appointment_datetime",
    "prediction_cutoff_datetime"
]

for column in datetime_columns:

    assert column in df_fe.columns, (
        f"Missing datetime column: {column}"
    )

    df_fe[column] = pd.to_datetime(
        df_fe[column],
        errors="coerce"
    )

    assert df_fe[column].notna().all(), (
        f"Invalid or missing dates in {column}"
    )

# Check-in time is not required for prediction.
# Parse it only for the missingness and leakage audit.

if "clinic_checkin_time" in df_fe.columns:

    df_fe["clinic_checkin_time"] = pd.to_datetime(
        df_fe["clinic_checkin_time"],
        errors="coerce"
    )

print("Datetime parsing: PASS")


# 7. Inspect the data structure.

print("\n4. DATA TYPES AND STRUCTURE")

column_structure = pd.DataFrame({
    "column": df_fe.columns,
    "data_type": [
        str(df_fe[column].dtype)
        for column in df_fe.columns
    ],
    "non_null_count": [
        int(df_fe[column].notna().sum())
        for column in df_fe.columns
    ],
    "unique_values": [
        int(df_fe[column].nunique(dropna=True))
        for column in df_fe.columns
    ]
})

display(column_structure)


# 8. Review missing values.
# No imputation is performed at this stage.

print("\n5. MISSING VALUE SUMMARY")

missing_summary = pd.DataFrame({
    "column": df_fe.columns,
    "missing_count": [
        int(df_fe[column].isna().sum())
        for column in df_fe.columns
    ],
    "missing_percent": [
        round(
            df_fe[column].isna().mean() * 100,
            2
        )
        for column in df_fe.columns
    ]
})

missing_summary = (
    missing_summary[
        missing_summary["missing_count"] > 0
    ]
    .sort_values(
        "missing_count",
        ascending=False
    )
    .reset_index(drop=True)
)

display(missing_summary)

print(
    "Missing values will be addressed later "
    "using training-only preprocessing."
)


# 9. Check appointment identifiers.

print("\n6. APPOINTMENT IDENTIFIER VALIDATION")

assert "appointment_id" in df_fe.columns
assert "patient_id" in df_fe.columns

missing_appointment_ids = int(
    df_fe["appointment_id"].isna().sum()
)

duplicate_appointment_ids = int(
    df_fe["appointment_id"].duplicated().sum()
)

print(
    f"Missing appointment IDs: "
    f"{missing_appointment_ids}"
)

print(
    f"Duplicate appointment IDs: "
    f"{duplicate_appointment_ids}"
)

assert missing_appointment_ids == 0
assert duplicate_appointment_ids == 0

print("Appointment identifier validation: PASS")


# 10. Validate booking and appointment chronology.

print("\n7. BOOKING CHRONOLOGY VALIDATION")

actual_lead_days = (
    (
        df_fe["appointment_datetime"]
        - df_fe["booked_datetime"]
    ).dt.total_seconds() / 86400
)

assert (actual_lead_days > 0).all(), (
    "Invalid booking chronology detected."
)

valid_cutoff_chronology = (
    (
        df_fe["prediction_cutoff_datetime"]
        >= df_fe["booked_datetime"]
    )
    &
    (
        df_fe["prediction_cutoff_datetime"]
        < df_fe["appointment_datetime"]
    )
).all()

assert valid_cutoff_chronology, (
    "Invalid prediction cutoff chronology detected."
)

print("Booking chronology: PASS")
print("Prediction cutoff chronology: PASS")


# 11. Reconstruct booking_notice_group.


print("\n8. BOOKING NOTICE GROUP RECONSTRUCTION")

short_notice_mask = actual_lead_days < 7

derived_notice_group = np.where(
    short_notice_mask,
    "Short notice",
    "Long notice"
)

if "booking_notice_group" not in df_fe.columns:

    df_fe["booking_notice_group"] = (
        derived_notice_group
    )

    print(
        "Missing booking_notice_group "
        "successfully reconstructed."
    )

else:

    # Check the existing values against the
    # timestamp-derived classification.

    existing_group = (
        df_fe["booking_notice_group"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    existing_is_short = existing_group.str.startswith(
        "short",
        na=False
    )

    existing_is_long = existing_group.str.startswith(
        "long",
        na=False
    )

    recognised_groups = (
        existing_is_short | existing_is_long
    ).all()

    assert recognised_groups, (
        "Unrecognised booking notice categories."
    )

    group_matches = (
        existing_is_short.to_numpy()
        == short_notice_mask.to_numpy()
    ).all()

    assert group_matches, (
        "Existing booking_notice_group differs "
        "from timestamp-derived notice groups."
    )

    print(
        "Existing booking_notice_group "
        "matches the timestamp-derived groups."
    )

notice_group_summary = (
    df_fe["booking_notice_group"]
    .value_counts(dropna=False)
    .rename_axis("booking_notice_group")
    .reset_index(name="appointments")
)

display(notice_group_summary)

print(
    "Total appointments:",
    int(notice_group_summary["appointments"].sum())
)

assert (
    notice_group_summary["appointments"].sum()
    == actual_rows
)

print("Booking notice group validation: PASS")


# 12. Validate the prediction cutoff.
#
# Short notice: predict at booking.
# Long notice: predict seven days before appointment.

print("\n9. PREDICTION CUTOFF VALIDATION")

expected_cutoff = (
    df_fe["booked_datetime"].copy()
)

expected_cutoff.loc[~short_notice_mask] = (
    df_fe.loc[
        ~short_notice_mask,
        "appointment_datetime"
    ]
    - pd.Timedelta(days=7)
)

cutoff_difference_seconds = (
    df_fe["prediction_cutoff_datetime"]
    - expected_cutoff
).dt.total_seconds().abs()

cutoff_valid = (
    cutoff_difference_seconds <= 1
)

valid_cutoff_count = int(
    cutoff_valid.sum()
)

invalid_cutoff_count = int(
    (~cutoff_valid).sum()
)

cutoff_summary = pd.DataFrame([
    {
        "prediction_group": "Short notice",
        "appointments": int(short_notice_mask.sum()),
        "prediction_rule": "At booking",
        "invalid_cutoffs": int(
            (~cutoff_valid.loc[short_notice_mask]).sum()
        )
    },
    {
        "prediction_group": "Long notice",
        "appointments": int((~short_notice_mask).sum()),
        "prediction_rule": "7 days before appointment",
        "invalid_cutoffs": int(
            (~cutoff_valid.loc[~short_notice_mask]).sum()
        )
    }
])

display(cutoff_summary)

print(f"Valid cutoffs: {valid_cutoff_count:,}")
print(f"Invalid cutoffs: {invalid_cutoff_count:,}")

assert cutoff_valid.all(), (
    f"{invalid_cutoff_count} invalid prediction "
    "cutoffs detected."
)

print("Prediction cutoff validation: PASS")


# 13. Check existing lead-time values.


print("\n10. BOOKING LEAD DAYS VALIDATION")

assert "booking_lead_days" in df_fe.columns

assert df_fe["booking_lead_days"].notna().all()

stored_lead_days = pd.to_numeric(
    df_fe["booking_lead_days"],
    errors="coerce"
)

assert stored_lead_days.notna().all()

lead_difference = (
    stored_lead_days - actual_lead_days
).abs()

lead_time_audit = pd.DataFrame([
    {
        "metric": "Maximum absolute difference (days)",
        "value": round(float(lead_difference.max()), 4)
    },
    {
        "metric": "Median absolute difference (days)",
        "value": round(float(lead_difference.median()), 4)
    },
    {
        "metric": "Stored short-notice appointments",
        "value": int((stored_lead_days < 7).sum())
    },
    {
        "metric": "Timestamp short-notice appointments",
        "value": int(short_notice_mask.sum())
    }
])

display(lead_time_audit)

notice_disagreement = int(
    (
        (stored_lead_days < 7)
        != short_notice_mask
    ).sum()
)

print(
    "Stored versus timestamp notice disagreements:",
    notice_disagreement
)



if notice_disagreement > 0:

    print(
        "WARNING: Stored lead days and exact timestamps "
        "produce different notice groups for some rows."
    )

else:

    print(
        "Stored and timestamp-derived notice groups agree."
    )


# 14. Check the existing prediction notice feature.

print("\n11. DAYS UNTIL APPOINTMENT VALIDATION")

assert (
    "days_until_appointment_at_prediction"
    in df_fe.columns
)

actual_days_until_prediction = (
    (
        df_fe["appointment_datetime"]
        - df_fe["prediction_cutoff_datetime"]
    ).dt.total_seconds() / 86400
)

stored_days_until_prediction = pd.to_numeric(
    df_fe["days_until_appointment_at_prediction"],
    errors="coerce"
)

assert stored_days_until_prediction.notna().all()

prediction_notice_difference = (
    stored_days_until_prediction
    - actual_days_until_prediction
).abs()

print(
    "Maximum absolute difference:",
    round(
        float(prediction_notice_difference.max()),
        4
    ),
    "days"
)

# A difference greater than one day is flagged.

if (prediction_notice_difference > 1).any():

    print(
        "WARNING: Some stored prediction notice "
        "values differ from exact timestamps "
        "by more than one day."
    )

else:

    print(
        "Prediction notice values are consistent "
        "within one day."
    )


# 15. Confirm required source columns.


print("\n12. REQUIRED FEATURE COLUMNS")

required_features = [
    "appointment_id",
    "patient_id",
    "age",
    "sex",
    "ethnic_group",
    "imd_decile",
    "appointment_type",
    "consultation_mode",
    "specialty",
    "referral_source",
    "booking_lead_days",
    "booking_notice_group",
    "days_until_appointment_at_prediction",
    "previous_dnas_12m",
    "rescheduled_count",
    "sms_reminder_sent",
    "dna_target"
]

missing_features = [
    column for column in required_features
    if column not in df_fe.columns
]

assert not missing_features, (
    f"Missing required columns: {missing_features}"
)

print(
    f"All {len(required_features)} required "
    "columns are available."
)


# 16. Document leakage-sensitive fields.

print("\n13. DATA LEAKAGE SAFEGUARDS")

leakage_policy = pd.DataFrame([
    {
        "feature": "dna_target",
        "decision": "Target only",
        "reason": "Outcome to be predicted."
    },
    {
        "feature": "attendance_status",
        "decision": "Exclude",
        "reason": "Reveals actual attendance outcome."
    },
    {
        "feature": "clinic_checkin_time",
        "decision": "Exclude",
        "reason": (
            "Check-in is an attendance-related "
            "event occurring after prediction."
        )
    },
    {
        "feature": "appointment_id",
        "decision": "Identifier only",
        "reason": "Not a predictive input."
    },
    {
        "feature": "patient_id",
        "decision": "Identifier only",
        "reason": (
            "Retain separately for tracking "
            "and repeated-patient evaluation."
        )
    },
    {
        "feature": "previous_dnas_12m",
        "decision": "Restricted",
        "reason": (
            "Historical count requires "
            "point-in-time verification."
        )
    },
    {
        "feature": "rescheduled_count",
        "decision": "Restricted",
        "reason": (
            "May include rescheduling events "
            "after prediction cutoff."
        )
    },
    {
        "feature": "sms_reminder_sent",
        "decision": "Restricted",
        "reason": (
            "Reminder timing relative to "
            "prediction cutoff is unverified."
        )
    }
])

display(leakage_policy)



if "clinic_checkin_time" in df_fe.columns:

    checkin_missing_by_outcome = (
        df_fe.groupby("dna_target")[
            "clinic_checkin_time"
        ]
        .apply(
            lambda series: round(
                series.isna().mean() * 100,
                2
            )
        )
        .reset_index(name="checkin_missing_percent")
    )

    display(checkin_missing_by_outcome)

print(
    "Leakage-sensitive fields documented. "
    "Final feature selection will occur in Cell 2."
)


# 17. Save Cell 1 audit reports.
# Do not overwrite the original modelling dataset.

print("\n14. SAVE FEATURE ENGINEERING AUDIT REPORTS")

cell01_reports = {
    "cell01_column_structure.csv": column_structure,
    "cell01_missing_values.csv": missing_summary,
    "cell01_population_summary.csv": population_summary,
    "cell01_booking_notice_summary.csv": notice_group_summary,
    "cell01_prediction_cutoff_audit.csv": cutoff_summary,
    "cell01_lead_time_audit.csv": lead_time_audit,
    "cell01_leakage_policy.csv": leakage_policy
}

for filename, table in cell01_reports.items():

    output_path = (
        FEATURE_REPORT_DIR / filename
    )

    table.to_csv(
        output_path,
        index=False
    )

    print(f"Saved: {output_path}")


# 18. Final validation and completion summary.

print("\n15. FINAL CELL 1 VALIDATION")

assert len(df_fe) == expected_rows
assert actual_dnas == expected_dnas
assert actual_attended == expected_attended

assert df_fe["appointment_id"].is_unique

assert cutoff_valid.all()

assert "booking_notice_group" in df_fe.columns

assert len(df_original) == len(df_fe)

print(f"Appointments: {len(df_fe):,}")
print(f"Original columns: {df_original.shape[1]}")
print(f"Working columns: {df_fe.shape[1]}")
print(f"Attended: {actual_attended:,}")
print(f"DNA: {actual_dnas:,}")
print(f"DNA rate: {dna_rate:.2f}%")

print(
    f"Booking notice groups: "
    f"{df_fe['booking_notice_group'].nunique()}"
)

print(
    f"Prediction cutoffs validated: "
    f"{valid_cutoff_count:,}/{len(df_fe):,}"
)

print(
    f"Audit reports saved: {len(cell01_reports)}"
)

print(
    "\nNo rows removed, no missing values imputed, "
    "and no models trained."
)

print("\nCELL 1 COMPLETED SUCCESSFULLY")
print("Ready for Cell 2: Leakage-safe feature selection.")


CELL 1: LOAD AND VALIDATE MODELLING DATASET
Pandas version: 3.1.1
NumPy version: 2.5.4

Project root: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction
Input dataset: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\data\processed\outpatient_dna_modelling.csv
Report directory: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering

1. LOAD MODELLING DATASET
Rows loaded: 36,264
Original columns: 24


,appointment_id,patient_id,age,sex,imd_decile,ethnic_group,distance_km,interpreter_required,long_term_conditions,specialty,...,booked_datetime,rescheduled_count,sms_reminder_sent,previous_dnas_12m,attendance_status,clinic_checkin_time,booking_lead_days,prediction_cutoff_datetime,days_until_appointment_at_prediction,dna_target
0,OPA2004425,P113753,70.0,Male,2.0,Not Stated,5.7,N,1,Respiratory Medicine,...,2025-06-07 12:59:00,1,Y,1,Attended,11:16,86.938194,2025-08-26 11:30:00,7.0,0
1,OPA2000901,P119960,58.0,Female,2.0,White,4.7,N,0,Cardiology,...,2025-08-13 12:33:00,0,Y,0,Attended,10:13,90.914583,2025-11-05 10:30:00,7.0,0
2,OPA2002727,P113052,69.0,Male,5.0,White,13.1,N,3,Trauma and Orthopaedics,...,2025-06-05 16:40:00,1,Y,0,Attended,14:22,38.909722,2025-07-07 14:30:00,7.0,0
3,OPA2007989,P120866,72.0,Female,9.0,White,4.1,N,1,Ophthalmology,...,2025-08-27 17:14:00,0,Y,0,Attended,09:38,46.698611,2025-10-06 10:00:00,7.0,0
4,OPA2023691,P113013,29.0,Female,7.0,White,16.2,N,0,Respiratory Medicine,...,2025-05-22 12:30:00,0,Y,1,Attended,11:34,33.958333,2025-06-18 11:30:00,7.0,0



2. MODELLING POPULATION VALIDATION


,outcome,appointments,percentage
0,Attended,32958,90.88
1,DNA,3306,9.12


Overall DNA rate: 9.12%
Population validation: PASS

3. DATETIME VALIDATION
Datetime parsing: PASS

4. DATA TYPES AND STRUCTURE


C:\Users\User.DESKTOP-775\AppData\Local\Temp\ipykernel_46444\1563227397.py:215: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_fe["clinic_checkin_time"] = pd.to_datetime(


... (9 outputs left out)

Stored versus timestamp notice disagreements: 0
Stored and timestamp-derived notice groups agree.

11. DAYS UNTIL APPOINTMENT VALIDATION
Maximum absolute difference: 0.0 days
Prediction notice values are consistent within one day.

12. REQUIRED FEATURE COLUMNS
All 17 required columns are available.

13. DATA LEAKAGE SAFEGUARDS


,feature,decision,reason
0,dna_target,Target only,Outcome to be predicted.
1,attendance_status,Exclude,Reveals actual attendance outcome.
2,clinic_checkin_time,Exclude,Check-in is an attendance-related event occurr...
3,appointment_id,Identifier only,Not a predictive input.
4,patient_id,Identifier only,Retain separately for tracking and repeated-pa...
5,previous_dnas_12m,Restricted,Historical count requires point-in-time verifi...
6,rescheduled_count,Restricted,May include rescheduling events after predicti...
7,sms_reminder_sent,Restricted,Reminder timing relative to prediction cutoff ...


,dna_target,checkin_missing_percent
0,0,0.0
1,1,100.0


Leakage-sensitive fields documented. Final feature selection will occur in Cell 2.

14. SAVE FEATURE ENGINEERING AUDIT REPORTS
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell01_column_structure.csv
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell01_missing_values.csv
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell01_population_summary.csv
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell01_booking_notice_summary.csv
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell01_prediction_cutoff_audit.csv
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell01_lead_time_audit.csv
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell01_leakage

In [11]:

# LEAKAGE-SAFE FEATURE SELECTION


# Objectives:
# 1. Audit every available column.
# 2. Identify outcome leakage and identifiers.
# 3. Restrict unverified historical features.
# 4. Define provisional predictive features.
# 5. Retain fairness variables separately.
# 6. Save the feature selection decisions.
#
# This cell does not train a model or modify
# the original dataset.

print("CELL 2: LEAKAGE-SAFE FEATURE SELECTION")


# 1. Confirm Cell 1 has completed.

print("\n1. VERIFY INPUTS")

assert "df_fe" in globals(), (
    "Run Cell 1 before Cell 2."
)

assert "FEATURE_REPORT_DIR" in globals()

assert len(df_fe) == 36264

assert df_fe["appointment_id"].is_unique

print(f"Appointments: {len(df_fe):,}")
print(f"Available columns: {df_fe.shape[1]}")


# 2. Define the feature governance categories.
#
# PROVISIONAL:
# Potentially available at prediction time,
# but source-system timing still needs review.
#
# CONDITIONAL:
# Must not enter the initial model without
# additional evidence or justification.
#
# EXCLUDE:
# Contains outcome information, identifiers,
# or future information.
#
# METADATA:
# Retained for splitting, auditing or reporting.

print("\n2. DEFINE FEATURE GOVERNANCE RULES")

feature_policy = {

    # Outcome information

    "dna_target": (
        "Target",
        "Prediction outcome; never an input feature."
    ),

    "attendance_status": (
        "Exclude",
        "Actual attendance outcome."
    ),

    "clinic_checkin_time": (
        "Exclude",
        "Post-prediction attendance information."
    ),

    # Identifiers

    "appointment_id": (
        "Metadata",
        "Appointment tracking only."
    ),

    "patient_id": (
        "Metadata",
        "Patient grouping and evaluation only."
    ),

    # Prediction and event timestamps

    "booked_datetime": (
        "Metadata",
        "Used to construct timing features."
    ),

    "appointment_datetime": (
        "Metadata",
        "Used to construct appointment features."
    ),

    "prediction_cutoff_datetime": (
        "Metadata",
        "Defines the information availability cutoff."
    ),

    # Patient characteristics

    "age": (
        "Provisional",
        "Potentially known before appointment."
    ),

    "sex": (
        "Conditional",
        "Retain for fairness; predictive use "
        "requires separate justification."
    ),

    "ethnic_group": (
        "Conditional",
        "Retain for fairness; predictive use "
        "requires separate justification."
    ),

    "imd_decile": (
        "Conditional",
        "Retain for deprivation fairness analysis; "
        "predictive use requires justification."
    ),

    "distance_km": (
        "Provisional",
        "Verify the distance was calculable "
        "using information available at cutoff."
    ),

    "interpreter_required": (
        "Provisional",
        "Verify the requirement was recorded "
        "before prediction."
    ),

    "long_term_conditions": (
        "Provisional",
        "Verify condition counts reflect "
        "information known at prediction."
    ),

    # Appointment characteristics

    "appointment_type": (
        "Provisional",
        "Verify appointment type at cutoff."
    ),

    "consultation_mode": (
        "Provisional",
        "Verify consultation mode at cutoff."
    ),

    "specialty": (
        "Provisional",
        "Verify specialty at cutoff."
    ),

    "referral_source": (
        "Provisional",
        "Verify referral source at cutoff."
    ),

    # Booking and prediction timing

    "booking_lead_days": (
        "Provisional",
        "Derived from booking and appointment dates; "
        "verify appointment date at cutoff."
    ),

    "booking_notice_group": (
        "Provisional",
        "Derived from booking lead time."
    ),

    "days_until_appointment_at_prediction": (
        "Provisional",
        "Derived from appointment and cutoff dates."
    ),

    # Historical and intervention features

    "previous_dnas_12m": (
        "Conditional",
        "Historical events may extend beyond cutoff."
    ),

    "rescheduled_count": (
        "Conditional",
        "Rescheduling may occur after cutoff."
    ),

    "sms_reminder_sent": (
        "Exclude",
        "Reminder status may reflect a later "
        "intervention and its timing is unverified."
    )
}


# 3. Audit every dataset column.

print("\n3. AUDIT ALL DATASET COLUMNS")

audit_rows = []

for column in df_fe.columns:

    if column in feature_policy:

        status, reason = feature_policy[column]

    else:

        status = "Unclassified"
        reason = "Manual review required."

    audit_rows.append({
        "feature": column,
        "data_type": str(df_fe[column].dtype),
        "missing_count": int(
            df_fe[column].isna().sum()
        ),
        "missing_percent": round(
            df_fe[column].isna().mean() * 100,
            2
        ),
        "governance_status": status,
        "reason": reason
    })

feature_governance_audit = pd.DataFrame(
    audit_rows
)

display(feature_governance_audit)

unclassified = feature_governance_audit.loc[
    feature_governance_audit[
        "governance_status"
    ] == "Unclassified"
]

assert unclassified.empty, (
    "Some features are unclassified. "
    "Review the feature governance audit."
)

assert feature_governance_audit["feature"].is_unique

assert (
    len(feature_governance_audit)
    == df_fe.shape[1]
)

print("Every column has been classified.")


# 4. Summarise feature classifications.

print("\n4. FEATURE CLASSIFICATION SUMMARY")

feature_status_summary = (
    feature_governance_audit[
        "governance_status"
    ]
    .value_counts()
    .rename_axis("governance_status")
    .reset_index(name="feature_count")
)

display(feature_status_summary)


# 5. Define provisional initial predictors.
#
# These exclude identifiers, outcomes, future
# information and restricted historical fields.


print("\n5. PROVISIONAL INITIAL PREDICTORS")

provisional_features = (
    feature_governance_audit.loc[
        feature_governance_audit[
            "governance_status"
        ] == "Provisional",
        "feature"
    ]
    .tolist()
)

print(
    f"Provisional predictors: "
    f"{len(provisional_features)}"
)

for feature in provisional_features:

    print(f" - {feature}")


# 6. Define conditional features.


print("\n6. CONDITIONAL FEATURES")

conditional_features = (
    feature_governance_audit.loc[
        feature_governance_audit[
            "governance_status"
        ] == "Conditional",
        "feature"
    ]
    .tolist()
)

for feature in conditional_features:

    print(f" - {feature}")


# 7. Define outcome, excluded and metadata fields.

print("\n7. EXCLUDED FIELDS AND METADATA")

excluded_features = (
    feature_governance_audit.loc[
        feature_governance_audit[
            "governance_status"
        ] == "Exclude",
        "feature"
    ]
    .tolist()
)

metadata_features = (
    feature_governance_audit.loc[
        feature_governance_audit[
            "governance_status"
        ] == "Metadata",
        "feature"
    ]
    .tolist()
)

print("\nExcluded fields:")

for feature in excluded_features:

    print(f" - {feature}")

print("\nMetadata fields:")

for feature in metadata_features:

    print(f" - {feature}")


# 8. Retain fairness information separately.
# Fairness variables are needed for subgroup.

print("\n8. FAIRNESS AUDIT VARIABLES")

fairness_features = [
    "age",
    "sex",
    "ethnic_group",
    "imd_decile"
]

assert all(
    feature in df_fe.columns
    for feature in fairness_features
)

df_fairness = df_fe[
    [
        "appointment_id",
        "patient_id",
        "dna_target"
    ] + fairness_features
].copy()

print(
    "Fairness audit rows:",
    len(df_fairness)
)

display(df_fairness.head())


# 9. Create a separate provisional feature view.



print("\n9. CREATE PROVISIONAL FEATURE VIEW")

X_provisional = df_fe[
    provisional_features
].copy()

y_target = df_fe[
    "dna_target"
].copy()

prediction_metadata = df_fe[
    [
        "appointment_id",
        "patient_id",
        "booked_datetime",
        "appointment_datetime",
        "prediction_cutoff_datetime"
    ]
].copy()

assert len(X_provisional) == len(y_target)

assert len(X_provisional) == len(
    prediction_metadata
)

assert "dna_target" not in X_provisional.columns

assert "clinic_checkin_time" not in (
    X_provisional.columns
)

assert "previous_dnas_12m" not in (
    X_provisional.columns
)

assert "rescheduled_count" not in (
    X_provisional.columns
)

assert "sms_reminder_sent" not in (
    X_provisional.columns
)

print(
    f"Provisional feature matrix: "
    f"{X_provisional.shape}"
)

print(
    f"Target rows: {len(y_target):,}"
)

print(
    "Restricted historical and outcome "
    "fields are excluded."
)


# 10. Document remaining approval requirements.

print("\n10. POINT-IN-TIME APPROVAL REQUIREMENTS")

feature_approval_plan = pd.DataFrame([
    {
        "feature_area": "Appointment characteristics",
        "requirement": (
            "Verify values at prediction cutoff."
        ),
        "status": "Pending"
    },
    {
        "feature_area": "Patient characteristics",
        "requirement": (
            "Verify source-system availability "
            "and appropriate predictive use."
        ),
        "status": "Pending"
    },
    {
        "feature_area": "Historical DNA counts",
        "requirement": (
            "Reconstruct counts using only events "
            "known before prediction cutoff."
        ),
        "status": "Restricted"
    },
    {
        "feature_area": "Rescheduling history",
        "requirement": (
            "Verify rescheduling event timestamps."
        ),
        "status": "Restricted"
    },
    {
        "feature_area": "SMS reminders",
        "requirement": (
            "Verify intervention timing; "
            "exclude from initial model."
        ),
        "status": "Excluded"
    },
    {
        "feature_area": "Fairness variables",
        "requirement": (
            "Retain for subgroup evaluation "
            "and justify predictive use."
        ),
        "status": "Separate audit"
    },
    {
        "feature_area": "Train/test splitting",
        "requirement": (
            "Use chronological splitting and "
            "fit preprocessing on training only."
        ),
        "status": "Pending"
    }
])

display(feature_approval_plan)


# 11. Save feature governance reports.

print("\n11. SAVE CELL 2 REPORTS")

cell02_reports = {
    "cell02_feature_governance_audit.csv":
        feature_governance_audit,

    "cell02_feature_status_summary.csv":
        feature_status_summary,

    "cell02_feature_approval_plan.csv":
        feature_approval_plan
}

for filename, table in cell02_reports.items():

    output_path = (
        FEATURE_REPORT_DIR / filename
    )

    table.to_csv(
        output_path,
        index=False
    )

    print(f"Saved: {output_path}")


# 12. Final validation.

print("\n12. FINAL CELL 2 VALIDATION")

all_selected_features = (
    provisional_features
    + conditional_features
    + excluded_features
    + metadata_features
    + ["dna_target"]
)

assert len(all_selected_features) == (
    len(set(all_selected_features))
)

assert set(all_selected_features) == set(
    df_fe.columns
)

assert len(df_fe) == 36264

assert len(df_fairness) == 36264

print(
    f"Columns classified: "
    f"{len(feature_governance_audit)}"
)

print(
    f"Provisional predictors: "
    f"{len(provisional_features)}"
)

print(
    f"Conditional features: "
    f"{len(conditional_features)}"
)

print(
    f"Excluded fields: "
    f"{len(excluded_features)}"
)

print(
    f"Metadata fields: "
    f"{len(metadata_features)}"
)

print(
    f"Fairness audit variables: "
    f"{len(fairness_features)}"
)

print(
    f"Reports saved: {len(cell02_reports)}"
)

print(
    "\nNo rows removed, no models trained, "
    "and no restricted historical features "
    "added to the initial predictor list."
)

print("COMPLETED SUCCESSFULLY")


CELL 2: LEAKAGE-SAFE FEATURE SELECTION

1. VERIFY INPUTS
Appointments: 36,264
Available columns: 25

2. DEFINE FEATURE GOVERNANCE RULES

3. AUDIT ALL DATASET COLUMNS


,feature,data_type,missing_count,missing_percent,governance_status,reason
0,appointment_id,str,0,0.00,Metadata,Appointment tracking only.
1,patient_id,str,0,0.00,Metadata,Patient grouping and evaluation only.
2,age,float64,48,0.13,Provisional,Potentially known before appointment.
3,sex,str,0,0.00,Conditional,Retain for fairness; predictive use requires s...
4,imd_decile,float64,1265,3.49,Conditional,Retain for deprivation fairness analysis; pred...
5,ethnic_group,str,0,0.00,Conditional,Retain for fairness; predictive use requires s...
6,distance_km,float64,362,1.00,Provisional,Verify the distance was calculable using infor...
7,interpreter_required,str,0,0.00,Provisional,Verify the requirement was recorded before pre...
8,long_term_conditions,int64,0,0.00,Provisional,Verify condition counts reflect information kn...
9,specialty,str,0,0.00,Provisional,Verify specialty at cutoff.


Every column has been classified.

4. FEATURE CLASSIFICATION SUMMARY


,governance_status,feature_count
0,Provisional,11
1,Metadata,5
2,Conditional,5
3,Exclude,3
4,Target,1



5. PROVISIONAL INITIAL PREDICTORS
Provisional predictors: 11
 - age
 - distance_km
 - interpreter_required
 - long_term_conditions
 - specialty
 - appointment_type
 - consultation_mode
 - referral_source
 - booking_lead_days
 - days_until_appointment_at_prediction
 - booking_notice_group

6. CONDITIONAL FEATURES
 - sex
 - imd_decile
 - ethnic_group
 - rescheduled_count
 - previous_dnas_12m

7. EXCLUDED FIELDS AND METADATA

Excluded fields:
 - sms_reminder_sent
 - attendance_status
 - clinic_checkin_time

Metadata fields:
 - appointment_id
 - patient_id
 - appointment_datetime
 - booked_datetime
 - prediction_cutoff_datetime

8. FAIRNESS AUDIT VARIABLES
Fairness audit rows: 36264


,appointment_id,patient_id,dna_target,age,sex,ethnic_group,imd_decile
0,OPA2004425,P113753,0,70.0,Male,Not Stated,2.0
1,OPA2000901,P119960,0,58.0,Female,White,2.0
2,OPA2002727,P113052,0,69.0,Male,White,5.0
3,OPA2007989,P120866,0,72.0,Female,White,9.0
4,OPA2023691,P113013,0,29.0,Female,White,7.0



9. CREATE PROVISIONAL FEATURE VIEW
Provisional feature matrix: (36264, 11)
Target rows: 36,264
Restricted historical and outcome fields are excluded.

10. POINT-IN-TIME APPROVAL REQUIREMENTS


,feature_area,requirement,status
0,Appointment characteristics,Verify values at prediction cutoff.,Pending
1,Patient characteristics,Verify source-system availability and appropri...,Pending
2,Historical DNA counts,Reconstruct counts using only events known bef...,Restricted
3,Rescheduling history,Verify rescheduling event timestamps.,Restricted
4,SMS reminders,Verify intervention timing; exclude from initi...,Excluded
5,Fairness variables,Retain for subgroup evaluation and justify pre...,Separate audit
6,Train/test splitting,Use chronological splitting and fit preprocess...,Pending



11. SAVE CELL 2 REPORTS
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell02_feature_governance_audit.csv
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell02_feature_status_summary.csv
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell02_feature_approval_plan.csv

12. FINAL CELL 2 VALIDATION
Columns classified: 25
Provisional predictors: 11
Conditional features: 5
Excluded fields: 3
Metadata fields: 5
Fairness audit variables: 4
Reports saved: 3

No rows removed, no models trained, and no restricted historical features added to the initial predictor list.
COMPLETED SUCCESSFULLY


In [12]:

# APPOINTMENT AND TIMING FEATURE ENGINEERING


# Objectives:
# 1. Validate required dates and feature inputs.
# 2. Engineer appointment calendar features.
# 3. Engineer appointment time-of-day features.
# 4. Engineer exact booking and prediction notice features.
# 5. Engineer interpretable lead-time bands.
# 6. Validate all new features.
# 7. Preserve the modelling population and target.
# 8. Save feature engineering audit reports.
#
# No rows removed, no imputation, no model training.


print("APPOINTMENT AND TIMING FEATURE ENGINEERING")


# 1. Verify previous cells.

print("\n1. VERIFY INPUT DATA")

required_objects = [
    "df_fe",
    "df_original",
    "provisional_features",
    "FEATURE_REPORT_DIR"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    f"Missing objects: {missing_objects}. "
    "Run Cells 1 and 2 first."
)

assert len(df_fe) == 36264

# Create a separate working copy so that
# the existing Cell 1 and Cell 2 data remain unchanged.

df_engineered = df_fe.copy(deep=True)

original_row_count = len(df_engineered)
original_column_count = df_engineered.shape[1]
original_target = df_engineered["dna_target"].copy()

print(f"Rows: {original_row_count:,}")
print(f"Starting columns: {original_column_count}")


# 2. Validate required timestamps.

print("\n2. VALIDATE TIMESTAMPS")

required_dates = [
    "booked_datetime",
    "appointment_datetime",
    "prediction_cutoff_datetime"
]

for column in required_dates:

    assert column in df_engineered.columns

    df_engineered[column] = pd.to_datetime(
        df_engineered[column],
        errors="coerce"
    )

    assert df_engineered[column].notna().all(), (
        f"Invalid timestamps in {column}"
    )

assert (
    df_engineered["booked_datetime"]
    < df_engineered["appointment_datetime"]
).all()

assert (
    (
        df_engineered["prediction_cutoff_datetime"]
        >= df_engineered["booked_datetime"]
    )
    &
    (
        df_engineered["prediction_cutoff_datetime"]
        < df_engineered["appointment_datetime"]
    )
).all()

print("Required timestamps validated.")


# 3. Calculate exact booking lead and notice.
# Use timestamp differences rather than rounded
# source columns.

print("\n3. ENGINEER EXACT TIMING FEATURES")

df_engineered["booking_lead_days_exact"] = (
    (
        df_engineered["appointment_datetime"]
        - df_engineered["booked_datetime"]
    ).dt.total_seconds() / 86400
)

df_engineered["prediction_notice_days_exact"] = (
    (
        df_engineered["appointment_datetime"]
        - df_engineered["prediction_cutoff_datetime"]
    ).dt.total_seconds() / 86400
)

df_engineered["is_short_notice"] = (
    df_engineered["booking_lead_days_exact"] < 7
).astype(int)

assert (
    df_engineered["booking_lead_days_exact"] > 0
).all()

assert (
    df_engineered["prediction_notice_days_exact"] > 0
).all()

print("Exact timing features created.")


# 4. Engineer appointment calendar features.
# Monday = 0, Sunday = 6.

print("\n4. ENGINEER CALENDAR FEATURES")

appointment_dates = (
    df_engineered["appointment_datetime"]
)

df_engineered["appointment_day_of_week"] = (
    appointment_dates.dt.dayofweek
)

df_engineered["appointment_month"] = (
    appointment_dates.dt.month
)

df_engineered["appointment_day_of_month"] = (
    appointment_dates.dt.day
)

df_engineered["appointment_is_weekend"] = (
    appointment_dates.dt.dayofweek >= 5
).astype(int)

# Calendar features are derived from the scheduled
# appointment date. 

print("Appointment calendar features created.")


# 5. Engineer appointment time-of-day features.

print("\n5. ENGINEER APPOINTMENT TIME FEATURES")

df_engineered["appointment_hour"] = (
    appointment_dates.dt.hour
)

df_engineered["appointment_minute"] = (
    appointment_dates.dt.minute
)

# Use a continuous representation of the hour.

df_engineered["appointment_time_decimal"] = (
    df_engineered["appointment_hour"]
    + df_engineered["appointment_minute"] / 60
)

# Define interpretable time periods.

def classify_appointment_period(hour):
    """
    Classify an appointment based on its hour.
    """

    if hour < 12:
        return "Morning"

    elif hour < 17:
        return "Afternoon"

    else:
        return "Evening"


df_engineered["appointment_period"] = (
    df_engineered["appointment_hour"]
    .apply(classify_appointment_period)
)

print("Appointment time features created.")


# 6. Engineer lead-time bands.
# Bands are descriptive and do not use the target.

print("\n6. ENGINEER BOOKING LEAD-TIME BANDS")

lead_bins = [
    -np.inf,
    7,
    14,
    30,
    60,
    90,
    180,
    np.inf
]

lead_labels = [
    "<7 days",
    "7-<14 days",
    "14-<30 days",
    "30-<60 days",
    "60-<90 days",
    "90-<180 days",
    "180+ days"
]

df_engineered["booking_lead_band"] = pd.cut(
    df_engineered["booking_lead_days_exact"],
    bins=lead_bins,
    labels=lead_labels,
    right=False
)

assert (
    df_engineered["booking_lead_band"]
    .notna()
    .all()
)

print("Booking lead-time bands created.")


# 7. Engineer a log-transformed lead time.
# to help represent long-tailed lead times
# without changing the underlying source value.

print("\n7. ENGINEER LOG LEAD TIME")

df_engineered["booking_lead_log1p"] = (
    np.log1p(
        df_engineered["booking_lead_days_exact"]
    )
)

assert np.isfinite(
    df_engineered["booking_lead_log1p"]
).all()

print("Log booking lead time created.")


# 8. Validate the short-notice classification.

print("\n8. VALIDATE NOTICE CLASSIFICATION")

expected_short = (
    df_engineered["booking_lead_days_exact"] < 7
)

actual_short = (
    df_engineered["is_short_notice"] == 1
)

assert (
    expected_short == actual_short
).all()

notice_summary = (
    df_engineered["is_short_notice"]
    .value_counts()
    .rename_axis("is_short_notice")
    .reset_index(name="appointments")
)

notice_summary["notice_group"] = (
    notice_summary["is_short_notice"]
    .map({
        0: "Long notice",
        1: "Short notice"
    })
)

display(notice_summary)


# 9. Validate the prediction timing rule.

print("\n9. VALIDATE PREDICTION TIMING")

short_mask = (
    df_engineered["is_short_notice"] == 1
)

long_mask = ~short_mask

short_cutoff_valid = (
    (
        df_engineered.loc[
            short_mask,
            "prediction_cutoff_datetime"
        ]
        -
        df_engineered.loc[
            short_mask,
            "booked_datetime"
        ]
    ).dt.total_seconds().abs() <= 1
).all()

long_cutoff_valid = (
    (
        df_engineered.loc[
            long_mask,
            "prediction_cutoff_datetime"
        ]
        -
        (
            df_engineered.loc[
                long_mask,
                "appointment_datetime"
            ]
            - pd.Timedelta(days=7)
        )
    ).dt.total_seconds().abs() <= 1
).all()

assert short_cutoff_valid
assert long_cutoff_valid

print("Short-notice cutoff: PASS")
print("Long-notice cutoff: PASS")


# 10. Identify the newly engineered features.

print("\n10. REVIEW ENGINEERED FEATURES")

engineered_feature_columns = [
    "booking_lead_days_exact",
    "prediction_notice_days_exact",
    "is_short_notice",
    "appointment_day_of_week",
    "appointment_month",
    "appointment_day_of_month",
    "appointment_is_weekend",
    "appointment_hour",
    "appointment_minute",
    "appointment_time_decimal",
    "appointment_period",
    "booking_lead_band",
    "booking_lead_log1p"
]

assert all(
    column in df_engineered.columns
    for column in engineered_feature_columns
)

feature_review = pd.DataFrame({
    "feature": engineered_feature_columns,
    "data_type": [
        str(df_engineered[column].dtype)
        for column in engineered_feature_columns
    ],
    "missing_count": [
        int(df_engineered[column].isna().sum())
        for column in engineered_feature_columns
    ],
    "unique_values": [
        int(
            df_engineered[column].nunique(
                dropna=True
            )
        )
        for column in engineered_feature_columns
    ]
})

display(feature_review)


# 11. Review appointment period distribution.

print("\n11. APPOINTMENT PERIOD DISTRIBUTION")

appointment_period_summary = (
    df_engineered["appointment_period"]
    .value_counts()
    .rename_axis("appointment_period")
    .reset_index(name="appointments")
)

display(appointment_period_summary)


# 12. Review booking lead-time distribution.
# Descriptive only

print("\n12. BOOKING LEAD-TIME DISTRIBUTION")

lead_band_summary = (
    df_engineered["booking_lead_band"]
    .value_counts(sort=False)
    .rename_axis("booking_lead_band")
    .reset_index(name="appointments")
)

display(lead_band_summary)


# 13. Define the provisional expanded feature list.


print("\n13. CREATE EXPANDED FEATURE LIST")

expanded_provisional_features = list(
    dict.fromkeys(
        provisional_features
        + engineered_feature_columns
    )
)

# Confirm no outcome or restricted historical
# columns entered the expanded list.

forbidden_features = {
    "dna_target",
    "attendance_status",
    "clinic_checkin_time",
    "appointment_id",
    "patient_id",
    "previous_dnas_12m",
    "rescheduled_count",
    "sms_reminder_sent",
    "sex",
    "ethnic_group",
    "imd_decile"
}

assert not (
    forbidden_features
    & set(expanded_provisional_features)
), (
    "Restricted feature detected in provisional list."
)

X_engineered_provisional = df_engineered[
    expanded_provisional_features
].copy()

print(
    "Expanded provisional feature matrix:",
    X_engineered_provisional.shape
)

print(
    "Note: Redundant features will be reviewed "
    "before model development."
)


# 14. Validate population and target preservation.

print("\n14. VALIDATE DATA INTEGRITY")

assert len(df_engineered) == original_row_count

assert (
    df_engineered["appointment_id"].is_unique
)

assert (
    df_engineered["dna_target"]
    .equals(original_target)
)

assert (
    df_engineered["dna_target"].sum()
    == 3306
)

assert (
    df_engineered[
        engineered_feature_columns
    ].isna().sum().sum()
    == 0
)

print("Population unchanged: PASS")
print("Target unchanged: PASS")
print("New feature completeness: PASS")


# 15. Save feature engineering audit reports.

print("\n15. SAVE CELL 3 REPORTS")

cell03_reports = {
    "cell03_engineered_feature_audit.csv":
        feature_review,

    "cell03_notice_group_summary.csv":
        notice_summary,

    "cell03_appointment_period_summary.csv":
        appointment_period_summary,

    "cell03_booking_lead_band_summary.csv":
        lead_band_summary
}

for filename, table in cell03_reports.items():

    output_path = (
        FEATURE_REPORT_DIR / filename
    )

    table.to_csv(
        output_path,
        index=False
    )

    print(f"Saved: {output_path}")


# 16. Final Cell 3 summary.

print("\n16. FINAL CELL 3 SUMMARY")

print(f"Appointments: {len(df_engineered):,}")

print(
    f"Original working columns: "
    f"{original_column_count}"
)

print(
    f"New features: "
    f"{len(engineered_feature_columns)}"
)

print(
    f"Total columns after engineering: "
    f"{df_engineered.shape[1]}"
)

print(
    f"Provisional predictor columns: "
    f"{len(expanded_provisional_features)}"
)

print(
    f"DNA cases preserved: "
    f"{int(df_engineered['dna_target'].sum()):,}"
)

print(
    "\nNo rows removed, no imputation, "
    "and no models trained."
)

print("COMPLETED SUCCESSFULLY")



APPOINTMENT AND TIMING FEATURE ENGINEERING

1. VERIFY INPUT DATA
Rows: 36,264
Starting columns: 25

2. VALIDATE TIMESTAMPS
Required timestamps validated.

3. ENGINEER EXACT TIMING FEATURES
Exact timing features created.

4. ENGINEER CALENDAR FEATURES
Appointment calendar features created.

5. ENGINEER APPOINTMENT TIME FEATURES
Appointment time features created.

6. ENGINEER BOOKING LEAD-TIME BANDS
Booking lead-time bands created.

7. ENGINEER LOG LEAD TIME
Log booking lead time created.

8. VALIDATE NOTICE CLASSIFICATION


C:\Users\User.DESKTOP-775\AppData\Local\Temp\ipykernel_46444\1759129222.py:144: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  appointment_dates.dt.dayofweek
C:\Users\User.DESKTOP-775\AppData\Local\Temp\ipykernel_46444\1759129222.py:156: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  appointment_dates.dt.dayofweek >= 5


,is_short_notice,appointments,notice_group
0,0,33567,Long notice
1,1,2697,Short notice



9. VALIDATE PREDICTION TIMING
Short-notice cutoff: PASS
Long-notice cutoff: PASS

10. REVIEW ENGINEERED FEATURES


,feature,data_type,missing_count,unique_values
0,booking_lead_days_exact,float64,0,29591
1,prediction_notice_days_exact,float64,0,2143
2,is_short_notice,int64,0,2
3,appointment_day_of_week,int32,0,5
4,appointment_month,int32,0,12
5,appointment_day_of_month,int32,0,31
6,appointment_is_weekend,int64,0,1
7,appointment_hour,int32,0,9
8,appointment_minute,int32,0,2
9,appointment_time_decimal,float64,0,17



11. APPOINTMENT PERIOD DISTRIBUTION


,appointment_period,appointments
0,Afternoon,21247
1,Morning,15017



12. BOOKING LEAD-TIME DISTRIBUTION


,booking_lead_band,appointments
0,<7 days,2697
1,7-<14 days,3788
2,14-<30 days,8643
3,30-<60 days,11308
4,60-<90 days,5568
5,90-<180 days,3950
6,180+ days,310



13. CREATE EXPANDED FEATURE LIST
Expanded provisional feature matrix: (36264, 24)
Note: Redundant features will be reviewed before model development.

14. VALIDATE DATA INTEGRITY
Population unchanged: PASS
Target unchanged: PASS
New feature completeness: PASS

15. SAVE CELL 3 REPORTS
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell03_engineered_feature_audit.csv
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell03_notice_group_summary.csv
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell03_appointment_period_summary.csv
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell03_booking_lead_band_summary.csv

16. FINAL CELL 3 SUMMARY
Appointments: 36,264
Original working columns: 25
New features: 13
Total columns after engineering: 38
Provisional predictor columns: 24
DNA cases preserved:

In [13]:

# PATIENT AND APPOINTMENT FEATURE ENGINEERING


# Objectives:
# 1. Validate the output of Cell 3.
# 2. Engineer age-group features.
# 3. Engineer travel-distance features.
# 4. Engineer long-term-condition features.
# 5. Create appointment and consultation interactions.
# 6. Preserve missing values without imputation.
# 7. Validate the population and leakage safeguards.
# 8. Save feature engineering reports.
#
# No rows removed, no imputation, no model training.

print("PATIENT AND APPOINTMENT FEATURE ENGINEERING")


# 1. Verify Cell 3 inputs.

print("\n1. VERIFY INPUTS")

required_objects = [
    "df_engineered",
    "engineered_feature_columns",
    "expanded_provisional_features",
    "FEATURE_REPORT_DIR"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    f"Missing objects: {missing_objects}. "
    "Run Cells 1-3 first."
)

df_patient_features = df_engineered.copy(deep=True)

starting_rows = len(df_patient_features)
starting_columns = df_patient_features.shape[1]

assert starting_rows == 36264
assert df_patient_features["dna_target"].sum() == 3306
assert df_patient_features["appointment_id"].is_unique

original_target = df_patient_features["dna_target"].copy()

print(f"Starting rows: {starting_rows:,}")
print(f"Starting columns: {starting_columns}")


# 2. Validate numeric patient attributes.
# Convert invalid numeric text to missing values.


print("\n2. VALIDATE NUMERIC ATTRIBUTES")

numeric_sources = [
    "age",
    "distance_km",
    "long_term_conditions"
]

for column in numeric_sources:

    assert column in df_patient_features.columns

    df_patient_features[column] = pd.to_numeric(
        df_patient_features[column],
        errors="coerce"
    )

# Ages were previously cleaned. Check their range.

valid_age = (
    df_patient_features["age"].isna()
    |
    df_patient_features["age"].between(0, 120)
)

assert valid_age.all(), (
    "Unexpected invalid ages detected."
)

# Travel distance cannot be negative.

valid_distance = (
    df_patient_features["distance_km"].isna()
    |
    (df_patient_features["distance_km"] >= 0)
)

assert valid_distance.all(), (
    "Negative travel distance detected."
)

# Long-term condition counts cannot be negative.

valid_conditions = (
    df_patient_features["long_term_conditions"].isna()
    |
    (
        df_patient_features["long_term_conditions"]
        >= 0
    )
)

assert valid_conditions.all(), (
    "Negative long-term condition counts detected."
)

print("Numeric patient attributes validated.")


# 3. Engineer age groups.
# Missing ages remain explicitly labelled Unknown.


print("\n3. ENGINEER AGE FEATURES")

age_bins = [
    -np.inf,
    18,
    30,
    45,
    60,
    75,
    np.inf
]

age_labels = [
    "Under 18",
    "18-29",
    "30-44",
    "45-59",
    "60-74",
    "75+"
]

df_patient_features["age_group"] = pd.cut(
    df_patient_features["age"],
    bins=age_bins,
    labels=age_labels,
    right=False
)

df_patient_features["age_group"] = (
    df_patient_features["age_group"]
    .cat.add_categories(["Unknown"])
    .fillna("Unknown")
)

df_patient_features["age_missing_flag"] = (
    df_patient_features["age"].isna()
).astype(int)

age_group_summary = (
    df_patient_features["age_group"]
    .value_counts(sort=False)
    .rename_axis("age_group")
    .reset_index(name="appointments")
)

display(age_group_summary)


# 4. Engineer travel-distance features.
# Preserve missing distance values in numerical
# transformations; do not replace with zero.

print("\n4. ENGINEER TRAVEL-DISTANCE FEATURES")

df_patient_features["distance_log1p"] = (
    np.log1p(
        df_patient_features["distance_km"]
    )
)

df_patient_features["distance_missing_flag"] = (
    df_patient_features["distance_km"].isna()
).astype(int)

distance_bins = [
    -np.inf,
    5,
    10,
    20,
    40,
    np.inf
]

distance_labels = [
    "<5 km",
    "5-<10 km",
    "10-<20 km",
    "20-<40 km",
    "40+ km"
]

df_patient_features["distance_band"] = pd.cut(
    df_patient_features["distance_km"],
    bins=distance_bins,
    labels=distance_labels,
    right=False
)

df_patient_features["distance_band"] = (
    df_patient_features["distance_band"]
    .cat.add_categories(["Unknown"])
    .fillna("Unknown")
)

distance_summary = (
    df_patient_features["distance_band"]
    .value_counts(sort=False)
    .rename_axis("distance_band")
    .reset_index(name="appointments")
)

display(distance_summary)


# 5. Engineer long-term-condition features.


print("\n5. ENGINEER LONG-TERM-CONDITION FEATURES")

conditions = df_patient_features[
    "long_term_conditions"
]

df_patient_features["has_long_term_condition"] = (
    conditions.gt(0)
    .where(conditions.notna())
    .astype("Int64")
)

df_patient_features["multiple_long_term_conditions"] = (
    conditions.ge(2)
    .where(conditions.notna())
    .astype("Int64")
)

condition_bins = [
    -np.inf,
    1,
    2,
    3,
    np.inf
]

condition_labels = [
    "0 conditions",
    "1 condition",
    "2 conditions",
    "3+ conditions"
]

df_patient_features["condition_group"] = pd.cut(
    conditions,
    bins=condition_bins,
    labels=condition_labels,
    right=False
)

df_patient_features["condition_group"] = (
    df_patient_features["condition_group"]
    .cat.add_categories(["Unknown"])
    .fillna("Unknown")
)

condition_summary = (
    df_patient_features["condition_group"]
    .value_counts(sort=False)
    .rename_axis("condition_group")
    .reset_index(name="appointments")
)

display(condition_summary)


# 6. Create appointment interaction features.
# These combine existing appointment attributes
# without using attendance outcomes.

print("\n6. ENGINEER APPOINTMENT INTERACTIONS")

required_categories = [
    "appointment_type",
    "consultation_mode",
    "specialty",
    "appointment_period"
]

assert all(
    column in df_patient_features.columns
    for column in required_categories
)

def normalise_category(series):
    """
    Standardise category labels for feature creation.
    Missing values are explicitly marked Unknown.
    """

    return (
        series.astype("string")
        .fillna("Unknown")
        .str.strip()
        .replace("", "Unknown")
    )

appointment_type_clean = normalise_category(
    df_patient_features["appointment_type"]
)

consultation_mode_clean = normalise_category(
    df_patient_features["consultation_mode"]
)

specialty_clean = normalise_category(
    df_patient_features["specialty"]
)

period_clean = normalise_category(
    df_patient_features["appointment_period"]
)

# Combined consultation and appointment type.

df_patient_features["type_mode_combination"] = (
    appointment_type_clean
    + " | "
    + consultation_mode_clean
)

# Combined specialty and consultation mode.

df_patient_features["specialty_mode_combination"] = (
    specialty_clean
    + " | "
    + consultation_mode_clean
)

# Combined consultation mode and time period.

df_patient_features["mode_period_combination"] = (
    consultation_mode_clean
    + " | "
    + period_clean
)

interaction_summary = pd.DataFrame({
    "feature": [
        "type_mode_combination",
        "specialty_mode_combination",
        "mode_period_combination"
    ],
    "unique_combinations": [
        int(
            df_patient_features[column].nunique()
        )
        for column in [
            "type_mode_combination",
            "specialty_mode_combination",
            "mode_period_combination"
        ]
    ]
})

display(interaction_summary)


# 7. Register all new features.

print("\n7. REVIEW NEW FEATURES")

patient_engineered_features = [
    "age_group",
    "age_missing_flag",
    "distance_log1p",
    "distance_missing_flag",
    "distance_band",
    "has_long_term_condition",
    "multiple_long_term_conditions",
    "condition_group",
    "type_mode_combination",
    "specialty_mode_combination",
    "mode_period_combination"
]

assert all(
    column in df_patient_features.columns
    for column in patient_engineered_features
)

patient_feature_audit = pd.DataFrame({
    "feature": patient_engineered_features,
    "data_type": [
        str(df_patient_features[column].dtype)
        for column in patient_engineered_features
    ],
    "missing_count": [
        int(df_patient_features[column].isna().sum())
        for column in patient_engineered_features
    ],
    "unique_values": [
        int(
            df_patient_features[column].nunique(
                dropna=True
            )
        )
        for column in patient_engineered_features
    ]
})

display(patient_feature_audit)


# 8. Create the expanded provisional feature list.

print("\n8. UPDATE PROVISIONAL FEATURE LIST")

all_provisional_features = list(
    dict.fromkeys(
        expanded_provisional_features
        + patient_engineered_features
    )
)

forbidden_features = {
    "dna_target",
    "attendance_status",
    "clinic_checkin_time",
    "appointment_id",
    "patient_id",
    "booked_datetime",
    "appointment_datetime",
    "prediction_cutoff_datetime",
    "previous_dnas_12m",
    "rescheduled_count",
    "sms_reminder_sent",
    "sex",
    "ethnic_group",
    "imd_decile"
}

assert not (
    set(all_provisional_features)
    & forbidden_features
), (
    "Restricted feature detected in "
    "provisional predictors."
)

X_patient_provisional = df_patient_features[
    all_provisional_features
].copy()

print(
    "Expanded provisional feature matrix:",
    X_patient_provisional.shape
)


# 9. Check data integrity and missingness.

print("\n9. VALIDATE DATA INTEGRITY")

assert len(df_patient_features) == starting_rows

assert df_patient_features[
    "appointment_id"
].is_unique

assert df_patient_features[
    "dna_target"
].equals(original_target)

assert int(
    df_patient_features["dna_target"].sum()
) == 3306

assert int(
    df_patient_features["age_missing_flag"].sum()
) == int(
    df_patient_features["age"].isna().sum()
)

assert int(
    df_patient_features["distance_missing_flag"].sum()
) == int(
    df_patient_features["distance_km"].isna().sum()
)

# Check that missing distance values have not
# silently become zero in the log transformation.

assert (
    df_patient_features["distance_log1p"].isna()
    ==
    df_patient_features["distance_km"].isna()
).all()

print("Population unchanged: PASS")
print("Target unchanged: PASS")
print("Missingness preservation: PASS")


# 10. Document source availability restrictions.

print("\n10. POINT-IN-TIME FEATURE REVIEW")

patient_feature_restrictions = pd.DataFrame([
    {
        "feature_area": "Age and age groups",
        "requirement": (
            "Confirm age was calculated using "
            "information available at prediction."
        ),
        "status": "Provisional"
    },
    {
        "feature_area": "Travel distance",
        "requirement": (
            "Verify address and clinic location "
            "were known at prediction."
        ),
        "status": "Provisional"
    },
    {
        "feature_area": "Long-term conditions",
        "requirement": (
            "Confirm condition counts represent "
            "information available at cutoff."
        ),
        "status": "Provisional"
    },
    {
        "feature_area": "Appointment interactions",
        "requirement": (
            "Verify appointment type, mode and "
            "specialty were known at cutoff."
        ),
        "status": "Provisional"
    },
    {
        "feature_area": "Fairness attributes",
        "requirement": (
            "Retain separately for fairness "
            "evaluation."
        ),
        "status": "Separate audit"
    },
    {
        "feature_area": "Historical attendance",
        "requirement": (
            "Exclude until point-in-time "
            "reconstruction is possible."
        ),
        "status": "Restricted"
    }
])

display(patient_feature_restrictions)


# 11. Save Cell 4 reports.

print("\n11. SAVE FEATURE ENGINEERING REPORTS")

cell04_reports = {
    "cell04_patient_feature_audit.csv":
        patient_feature_audit,

    "cell04_age_group_summary.csv":
        age_group_summary,

    "cell04_distance_band_summary.csv":
        distance_summary,

    "cell04_condition_group_summary.csv":
        condition_summary,

    "cell04_appointment_interactions.csv":
        interaction_summary,

    "cell04_point_in_time_review.csv":
        patient_feature_restrictions
}

for filename, table in cell04_reports.items():

    output_path = FEATURE_REPORT_DIR / filename

    table.to_csv(
        output_path,
        index=False
    )

    print(f"Saved: {output_path}")


# 12. Final Cell 4 summary.

print("\n12. FINAL CELL 4 SUMMARY")

print(f"Appointments: {len(df_patient_features):,}")
print(f"Starting columns: {starting_columns}")
print(
    f"New features: "
    f"{len(patient_engineered_features)}"
)
print(
    f"Total columns: "
    f"{df_patient_features.shape[1]}"
)
print(
    f"Provisional predictors: "
    f"{len(all_provisional_features)}"
)
print(
    f"DNA cases: "
    f"{int(df_patient_features['dna_target'].sum()):,}"
)
print(
    f"Reports saved: {len(cell04_reports)}"
)

print(
    "\nNo rows removed, no missing values imputed, "
    "and no models trained."
)

print("COMPLETED SUCCESSFULLY")



PATIENT AND APPOINTMENT FEATURE ENGINEERING

1. VERIFY INPUTS
Starting rows: 36,264
Starting columns: 38

2. VALIDATE NUMERIC ATTRIBUTES
Numeric patient attributes validated.

3. ENGINEER AGE FEATURES


,age_group,appointments
0,Under 18,4416
1,18-29,4067
2,30-44,5067
3,45-59,5192
4,60-74,9075
5,75+,8399
6,Unknown,48



4. ENGINEER TRAVEL-DISTANCE FEATURES


,distance_band,appointments
0,<5 km,11607
1,5-<10 km,13372
2,10-<20 km,9331
3,20-<40 km,1580
4,40+ km,12
5,Unknown,362



5. ENGINEER LONG-TERM-CONDITION FEATURES


,condition_group,appointments
0,0 conditions,13771
1,1 condition,9712
2,2 conditions,6286
3,3+ conditions,6495
4,Unknown,0


... (3 outputs left out)

,feature,data_type,missing_count,unique_values
0,age_group,category,0,7
1,age_missing_flag,int64,0,2
2,distance_log1p,float64,362,348
3,distance_missing_flag,int64,0,2
4,distance_band,category,0,6
5,has_long_term_condition,Int64,0,2
6,multiple_long_term_conditions,Int64,0,2
7,condition_group,category,0,4
8,type_mode_combination,string,0,6
9,specialty_mode_combination,string,0,27



8. UPDATE PROVISIONAL FEATURE LIST
Expanded provisional feature matrix: (36264, 35)

9. VALIDATE DATA INTEGRITY
Population unchanged: PASS
Target unchanged: PASS
Missingness preservation: PASS

10. POINT-IN-TIME FEATURE REVIEW


,feature_area,requirement,status
0,Age and age groups,Confirm age was calculated using information a...,Provisional
1,Travel distance,Verify address and clinic location were known ...,Provisional
2,Long-term conditions,Confirm condition counts represent information...,Provisional
3,Appointment interactions,"Verify appointment type, mode and specialty we...",Provisional
4,Fairness attributes,Retain separately for fairness evaluation.,Separate audit
5,Historical attendance,Exclude until point-in-time reconstruction is ...,Restricted



11. SAVE FEATURE ENGINEERING REPORTS
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell04_patient_feature_audit.csv
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell04_age_group_summary.csv
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell04_distance_band_summary.csv
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell04_condition_group_summary.csv
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell04_appointment_interactions.csv
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell04_point_in_time_review.csv

12. FINAL CELL 4 SUMMARY
Appointments: 36,264
Starting columns: 38
New features: 11
Total columns: 49
Provisional predictors: 35
DNA cases: 3,306
Reports saved: 6

No rows removed, 

In [14]:

# FEATURE QUALITY AND REDUNDANCY REVIEW


# Objectives:

# 1. Examine missing values and data types.
# 2. Identify constant and near-constant features.
# 3. Review high-cardinality categorical features.
# 4. Identify strongly correlated numeric features.
# 5. Document known feature redundancy.
# 6. Prepare a compact candidate feature list.
# 7. Save feature quality reports.
#
# No outcome-based selection, imputation,
# model training or row removal is performed.

print("CELL 5: FEATURE QUALITY AND REDUNDANCY REVIEW")


# 1. Verify previous notebook outputs.

print("\n1. VERIFY INPUTS")

required_objects = [
    "df_patient_features",
    "all_provisional_features",
    "FEATURE_REPORT_DIR"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    f"Missing objects: {missing_objects}. "
    "Run Cells 1-4 first."
)

df_quality = df_patient_features.copy(deep=True)

assert len(df_quality) == 36264
assert df_quality["appointment_id"].is_unique
assert int(df_quality["dna_target"].sum()) == 3306

assert len(all_provisional_features) == 35

assert all(
    feature in df_quality.columns
    for feature in all_provisional_features
)

print(f"Appointments: {len(df_quality):,}")
print(f"Dataset columns: {df_quality.shape[1]}")
print(
    f"Provisional predictors: "
    f"{len(all_provisional_features)}"
)


# 2. Create the feature quality inventory.

print("\n2. FEATURE QUALITY INVENTORY")

quality_rows = []

for feature in all_provisional_features:

    series = df_quality[feature]

    non_missing = int(series.notna().sum())
    missing = int(series.isna().sum())
    unique = int(series.nunique(dropna=True))

    quality_rows.append({
        "feature": feature,
        "data_type": str(series.dtype),
        "missing_count": missing,
        "missing_percent": round(
            missing / len(df_quality) * 100,
            2
        ),
        "unique_values": unique,
        "unique_ratio": round(
            unique / max(non_missing, 1),
            4
        ),
        "constant_feature": unique <= 1
    })

feature_quality_audit = pd.DataFrame(
    quality_rows
)

display(
    feature_quality_audit.sort_values(
        ["missing_percent", "unique_values"],
        ascending=[False, False]
    )
)


# 3. Identify missingness concerns.
# Missing values are not imputed here.

print("\n3. MISSINGNESS REVIEW")

missingness_review = (
    feature_quality_audit.loc[
        feature_quality_audit["missing_count"] > 0,
        [
            "feature",
            "missing_count",
            "missing_percent"
        ]
    ]
    .sort_values(
        "missing_percent",
        ascending=False
    )
    .reset_index(drop=True)
)

display(missingness_review)

high_missing_features = (
    feature_quality_audit.loc[
        feature_quality_audit["missing_percent"] >= 30,
        "feature"
    ].tolist()
)

print(
    "Features with at least 30% missing values:",
    high_missing_features
)

print(
    "Missingness will be handled using "
    "training-only preprocessing."
)


# 4. Identify constant and near-constant features.

print("\n4. CONSTANT AND NEAR-CONSTANT FEATURES")

constant_features = (
    feature_quality_audit.loc[
        feature_quality_audit["constant_feature"],
        "feature"
    ].tolist()
)

near_constant_rows = []

for feature in all_provisional_features:

    counts = df_quality[feature].value_counts(
        dropna=False,
        normalize=True
    )

    dominant_share = (
        float(counts.iloc[0])
        if not counts.empty
        else 0.0
    )

    near_constant_rows.append({
        "feature": feature,
        "dominant_value_percent": round(
            dominant_share * 100,
            2
        ),
        "near_constant_99_percent": (
            dominant_share >= 0.99
        )
    })

near_constant_audit = pd.DataFrame(
    near_constant_rows
)

display(
    near_constant_audit.loc[
        near_constant_audit[
            "near_constant_99_percent"
        ]
    ]
)

print("Constant features:", constant_features)

# Near-constant features are flagged for review,
# not automatically removed.


# 5. Separate numeric and categorical features.

print("\n5. FEATURE DATA TYPE REVIEW")

numeric_features = [
    feature for feature in all_provisional_features
    if pd.api.types.is_numeric_dtype(
        df_quality[feature]
    )
]

categorical_features = [
    feature for feature in all_provisional_features
    if feature not in numeric_features
]

print(f"Numeric features: {len(numeric_features)}")
print(
    f"Categorical features: "
    f"{len(categorical_features)}"
)


# 6. Examine categorical cardinality.


print("\n6. CATEGORICAL CARDINALITY REVIEW")

categorical_rows = []

for feature in categorical_features:

    series = df_quality[feature]

    value_counts = series.value_counts(
        dropna=False
    )

    rare_categories = int(
        (value_counts < 50).sum()
    )

    categorical_rows.append({
        "feature": feature,
        "unique_categories": int(
            series.nunique(dropna=True)
        ),
        "rare_categories_under_50": rare_categories,
        "missing_count": int(series.isna().sum())
    })

categorical_cardinality = pd.DataFrame(
    categorical_rows
).sort_values(
    "unique_categories",
    ascending=False
)

display(categorical_cardinality)

high_cardinality_features = (
    categorical_cardinality.loc[
        categorical_cardinality[
            "unique_categories"
        ] > 30,
        "feature"
    ].tolist()
)

print(
    "Features with more than 30 categories:",
    high_cardinality_features
)

# Cardinality is descriptive.



# 7. Check numeric feature correlations.
# The target is NOT included.

print("\n7. NUMERIC FEATURE CORRELATION REVIEW")

numeric_matrix = df_quality[
    numeric_features
].apply(
    pd.to_numeric,
    errors="coerce"
)

correlation_matrix = numeric_matrix.corr(
    method="spearman"
)

correlation_pairs = []

for i, first_feature in enumerate(
    correlation_matrix.columns
):

    for second_feature in (
        correlation_matrix.columns[i + 1:]
    ):

        correlation = correlation_matrix.loc[
            first_feature,
            second_feature
        ]

        if pd.notna(correlation):

            correlation_pairs.append({
                "feature_1": first_feature,
                "feature_2": second_feature,
                "spearman_correlation": round(
                    float(correlation),
                    4
                ),
                "absolute_correlation": round(
                    abs(float(correlation)),
                    4
                )
            })

correlation_pairs_df = pd.DataFrame(
    correlation_pairs,
    columns=[
        "feature_1",
        "feature_2",
        "spearman_correlation",
        "absolute_correlation"
    ]
)

high_correlation_pairs = (
    correlation_pairs_df.loc[
        correlation_pairs_df[
            "absolute_correlation"
        ] >= 0.95
    ]
    .sort_values(
        "absolute_correlation",
        ascending=False
    )
    .reset_index(drop=True)
)

display(high_correlation_pairs)

print(
    "Numeric pairs with absolute Spearman "
    "correlation >= 0.95:",
    len(high_correlation_pairs)
)


# 8. Document known deterministic redundancy.


print("\n8. KNOWN FEATURE REDUNDANCY")

redundancy_plan = pd.DataFrame([
    {
        "feature_group": "Booking lead time",
        "preferred_feature": "booking_lead_days_exact",
        "review_or_remove": (
            "booking_lead_days, booking_lead_log1p, "
            "booking_lead_band"
        ),
        "reason": (
            "Alternative representations of "
            "the same booking interval."
        )
    },
    {
        "feature_group": "Notice classification",
        "preferred_feature": "is_short_notice",
        "review_or_remove": (
            "booking_notice_group, "
            "days_until_appointment_at_prediction, "
            "prediction_notice_days_exact"
        ),
        "reason": (
            "Closely related to the "
            "prediction timing rule."
        )
    },
    {
        "feature_group": "Appointment weekday",
        "preferred_feature": "appointment_day_of_week",
        "review_or_remove": "appointment_is_weekend",
        "reason": (
            "Weekend status is determined "
            "by day of week."
        )
    },
    {
        "feature_group": "Appointment time",
        "preferred_feature": "appointment_hour",
        "review_or_remove": (
            "appointment_minute, "
            "appointment_time_decimal, "
            "appointment_period"
        ),
        "reason": (
            "Overlapping representations "
            "of appointment time."
        )
    },
    {
        "feature_group": "Age",
        "preferred_feature": "age",
        "review_or_remove": "age_group",
        "reason": (
            "Age group is derived from age."
        )
    },
    {
        "feature_group": "Travel distance",
        "preferred_feature": "distance_km",
        "review_or_remove": (
            "distance_log1p, distance_band"
        ),
        "reason": (
            "Alternative transformations "
            "of travel distance."
        )
    },
    {
        "feature_group": "Long-term conditions",
        "preferred_feature": "long_term_conditions",
        "review_or_remove": (
            "has_long_term_condition, "
            "multiple_long_term_conditions, "
            "condition_group"
        ),
        "reason": (
            "Derived from the same "
            "condition count."
        )
    },
    {
        "feature_group": "Appointment interactions",
        "preferred_feature": (
            "appointment_type, consultation_mode, "
            "specialty, appointment_hour"
        ),
        "review_or_remove": (
            "type_mode_combination, "
            "specialty_mode_combination, "
            "mode_period_combination"
        ),
        "reason": (
            "Interaction terms may increase "
            "cardinality and complexity."
        )
    }
])

display(redundancy_plan)


# 9. Define a compact candidate feature list.


print("\n9. DEFINE COMPACT CANDIDATE FEATURES")

compact_candidate_features = [
    "age",
    "age_missing_flag",
    "distance_km",
    "distance_missing_flag",
    "long_term_conditions",
    "interpreter_required",
    "specialty",
    "appointment_type",
    "consultation_mode",
    "referral_source",
    "booking_lead_days_exact",
    "is_short_notice",
    "appointment_day_of_week",
    "appointment_month",
    "appointment_hour"
]

assert all(
    feature in all_provisional_features
    for feature in compact_candidate_features
)

assert len(compact_candidate_features) == len(
    set(compact_candidate_features)
)

compact_feature_view = df_quality[
    compact_candidate_features
].copy()

print(
    "Compact candidate feature matrix:",
    compact_feature_view.shape
)

display(compact_feature_view.head())


# 10. Validate leakage restrictions.

print("\n10. LEAKAGE SAFEGUARDS")

forbidden_features = {
    "dna_target",
    "attendance_status",
    "clinic_checkin_time",
    "appointment_id",
    "patient_id",
    "booked_datetime",
    "appointment_datetime",
    "prediction_cutoff_datetime",
    "previous_dnas_12m",
    "rescheduled_count",
    "sms_reminder_sent",
    "sex",
    "ethnic_group",
    "imd_decile"
}

assert not (
    set(compact_candidate_features)
    & forbidden_features
)

assert len(compact_feature_view) == 36264

assert int(df_quality["dna_target"].sum()) == 3306

print("Restricted fields excluded: PASS")
print("Population preserved: PASS")
print("Target preserved: PASS")


# 11. Document remaining source-availability work.

print("\n11. FEATURE READINESS REVIEW")

feature_readiness = pd.DataFrame([
    {
        "area": "Appointment calendar and time",
        "status": "Provisional",
        "requirement": (
            "Confirm appointment schedule "
            "as it existed at prediction cutoff."
        )
    },
    {
        "area": "Patient characteristics",
        "status": "Provisional",
        "requirement": (
            "Confirm age, distance, interpreter "
            "and condition information at cutoff."
        )
    },
    {
        "area": "Historical DNA and rescheduling",
        "status": "Restricted",
        "requirement": (
            "Reconstruct from historical "
            "events or exclude."
        )
    },
    {
        "area": "SMS reminders",
        "status": "Excluded",
        "requirement": (
            "Do not use without verified "
            "pre-cutoff intervention timing."
        )
    },
    {
        "area": "Fairness attributes",
        "status": "Separate audit",
        "requirement": (
            "Retain for subgroup evaluation."
        )
    },
    {
        "area": "Feature redundancy",
        "status": "Baseline prepared",
        "requirement": (
            "Compare compact and expanded "
            "feature sets on held-out data."
        )
    },
    {
        "area": "Chronological splitting",
        "status": "Pending",
        "requirement": (
            "Split before fitting imputation, "
            "encoding or model selection."
        )
    }
])

display(feature_readiness)


# 12. Save Cell 5 reports.

print("\n12. SAVE FEATURE QUALITY REPORTS")

cell05_reports = {
    "cell05_feature_quality_audit.csv":
        feature_quality_audit,

    "cell05_missingness_review.csv":
        missingness_review,

    "cell05_near_constant_audit.csv":
        near_constant_audit,

    "cell05_categorical_cardinality.csv":
        categorical_cardinality,

    "cell05_high_correlation_pairs.csv":
        high_correlation_pairs,

    "cell05_redundancy_plan.csv":
        redundancy_plan,

    "cell05_compact_candidate_features.csv":
        pd.DataFrame({
            "feature": compact_candidate_features
        }),

    "cell05_feature_readiness.csv":
        feature_readiness
}

for filename, table in cell05_reports.items():

    output_path = FEATURE_REPORT_DIR / filename

    table.to_csv(
        output_path,
        index=False
    )

    print(f"Saved: {output_path}")


# 13. Final validation.

print("\n13. FINAL CELL 5 VALIDATION")

assert len(df_quality) == 36264

assert df_quality["appointment_id"].is_unique

assert len(compact_candidate_features) == 15

assert len(all_provisional_features) == 35

assert "dna_target" not in compact_feature_view.columns

print(f"Appointments: {len(df_quality):,}")
print(f"Provisional features reviewed: 35")
print(
    f"Compact candidate features: "
    f"{len(compact_candidate_features)}"
)
print(
    f"High-correlation numeric pairs: "
    f"{len(high_correlation_pairs)}"
)
print(
    f"Constant features: {len(constant_features)}"
)
print(
    f"High-cardinality categorical features: "
    f"{len(high_cardinality_features)}"
)
print(f"Reports saved: {len(cell05_reports)}")

print(
    "\nNo rows removed, no imputation, "
    "no outcome-based feature selection "
    "and no models trained."
)

print("COMPLETED SUCCESSFULLY")



CELL 5: FEATURE QUALITY AND REDUNDANCY REVIEW

1. VERIFY INPUTS
Appointments: 36,264
Dataset columns: 49
Provisional predictors: 35

2. FEATURE QUALITY INVENTORY


,feature,data_type,missing_count,missing_percent,unique_values,unique_ratio,constant_feature
1,distance_km,float64,362,1.00,348,0.0097,False
26,distance_log1p,float64,362,1.00,348,0.0097,False
0,age,float64,48,0.13,102,0.0028,False
8,booking_lead_days,float64,0,0.00,29591,0.8160,False
11,booking_lead_days_exact,float64,0,0.00,29591,0.8160,False
23,booking_lead_log1p,float64,0,0.00,29591,0.8160,False
9,days_until_appointment_at_prediction,float64,0,0.00,2143,0.0591,False
12,prediction_notice_days_exact,float64,0,0.00,2143,0.0591,False
16,appointment_day_of_month,int32,0,0.00,31,0.0009,False
33,specialty_mode_combination,string,0,0.00,27,0.0007,False



3. MISSINGNESS REVIEW


,feature,missing_count,missing_percent
0,distance_km,362,1.00
1,distance_log1p,362,1.00
2,age,48,0.13


Features with at least 30% missing values: []
Missingness will be handled using training-only preprocessing.

4. CONSTANT AND NEAR-CONSTANT FEATURES


,feature,dominant_value_percent,near_constant_99_percent
17,appointment_is_weekend,100.00,True
25,age_missing_flag,99.87,True
27,distance_missing_flag,99.00,True


... (7 outputs left out)

,age,age_missing_flag,distance_km,distance_missing_flag,long_term_conditions,interpreter_required,specialty,appointment_type,consultation_mode,referral_source,booking_lead_days_exact,is_short_notice,appointment_day_of_week,appointment_month,appointment_hour
0,70.0,0,5.7,0,1,N,Respiratory Medicine,Follow-up,Video,Consultant,86.938194,0,1,9,11
1,58.0,0,4.7,0,0,N,Cardiology,Follow-up,Face-to-face,Consultant,90.914583,0,2,11,10
2,69.0,0,13.1,0,3,N,Trauma and Orthopaedics,Follow-up,Face-to-face,Consultant,38.909722,0,0,7,14
3,72.0,0,4.1,0,1,N,Ophthalmology,Follow-up,Face-to-face,Consultant,46.698611,0,0,10,10
4,29.0,0,16.2,0,0,N,Respiratory Medicine,Follow-up,Face-to-face,Consultant,33.958333,0,2,6,11



10. LEAKAGE SAFEGUARDS
Restricted fields excluded: PASS
Population preserved: PASS
Target preserved: PASS

11. FEATURE READINESS REVIEW


,area,status,requirement
0,Appointment calendar and time,Provisional,Confirm appointment schedule as it existed at ...
1,Patient characteristics,Provisional,"Confirm age, distance, interpreter and conditi..."
2,Historical DNA and rescheduling,Restricted,Reconstruct from historical events or exclude.
3,SMS reminders,Excluded,Do not use without verified pre-cutoff interve...
4,Fairness attributes,Separate audit,Retain for subgroup evaluation.
5,Feature redundancy,Baseline prepared,Compare compact and expanded feature sets on h...
6,Chronological splitting,Pending,"Split before fitting imputation, encoding or m..."



12. SAVE FEATURE QUALITY REPORTS
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell05_feature_quality_audit.csv
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell05_missingness_review.csv
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell05_near_constant_audit.csv
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell05_categorical_cardinality.csv
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell05_high_correlation_pairs.csv
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell05_redundancy_plan.csv
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell05_compact_candidate_features.csv
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-p

In [15]:

# CHRONOLOGICAL TRAIN / VALIDATION / TEST SPLIT


# Objectives:
# 1. Validate the Cell 5 feature set.
# 2. Sort appointments chronologically.
# 3. Create train, validation and test periods.
# 4. Avoid splitting appointments on the same date.
# 5. Check outcome maturity across split boundaries.
# 6. Retain identifiers and fairness attributes.
# 7. Verify no appointment appears in multiple splits.
# 8. Save reproducible split manifests.
#
# No preprocessing is fitted and no models are trained.

print("CELL 6: CHRONOLOGICAL DATA SPLITTING")

# 1. Validate inputs.

required_objects = [
    "df_quality",
    "compact_candidate_features",
    "FEATURE_REPORT_DIR",
    "PROJECT_ROOT"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    f"Run Cells 1-5 first. Missing: {missing_objects}"
)

df_split = df_quality.copy(deep=True)

assert len(df_split) == 36264
assert df_split["appointment_id"].is_unique
assert int(df_split["dna_target"].sum()) == 3306

assert all(
    feature in df_split.columns
    for feature in compact_candidate_features
)

print(f"Input appointments: {len(df_split):,}")
print(f"Candidate predictors: {len(compact_candidate_features)}")


# 2. Validate chronological information.

print("\n1. VALIDATE CHRONOLOGY")

date_columns = [
    "booked_datetime",
    "prediction_cutoff_datetime",
    "appointment_datetime"
]

for column in date_columns:

    df_split[column] = pd.to_datetime(
        df_split[column],
        errors="coerce"
    )

    assert df_split[column].notna().all()

assert (
    df_split["booked_datetime"]
    <= df_split["prediction_cutoff_datetime"]
).all()

assert (
    df_split["prediction_cutoff_datetime"]
    < df_split["appointment_datetime"]
).all()

print("Chronology: PASS")


# 3. Sort by appointment date.
# Appointment date is used because the attendance
# outcome is observed after the appointment.
#
# Prediction cutoff will be used separately to
# check whether earlier outcomes were available.

print("\n2. SORT APPOINTMENTS")

df_split = df_split.sort_values(
    [
        "appointment_datetime",
        "appointment_id"
    ]
).reset_index(drop=True)

df_split["appointment_date"] = (
    df_split["appointment_datetime"].dt.normalize()
)

print(
    "First appointment:",
    df_split["appointment_datetime"].min()
)

print(
    "Last appointment:",
    df_split["appointment_datetime"].max()
)


# 4. Select chronological date boundaries.
# Never divide a calendar date across two splits.

print("\n3. DEFINE SPLIT BOUNDARIES")

unique_dates = pd.Series(
    df_split["appointment_date"]
    .drop_duplicates()
    .sort_values()
    .to_numpy()
)

assert len(unique_dates) >= 3, (
    "At least three distinct appointment dates "
    "are required."
)

# Find the dates containing approximately 70%
# and 85% of the appointment records.

train_position = int(len(df_split) * 0.70)
validation_position = int(len(df_split) * 0.85)

train_boundary = df_split.loc[
    train_position,
    "appointment_date"
]

test_boundary = df_split.loc[
    validation_position,
    "appointment_date"
]

assert train_boundary < test_boundary, (
    "Chronological split boundaries overlap."
)

print("Validation period begins:", train_boundary)
print("Test period begins:", test_boundary)


# 5. Assign the three chronological periods.

print("\n4. ASSIGN SPLITS")

df_split["data_split"] = np.select(
    [
        df_split["appointment_date"] < train_boundary,
        (
            (df_split["appointment_date"] >= train_boundary)
            & (df_split["appointment_date"] < test_boundary)
        )
    ],
    [
        "train",
        "validation"
    ],
    default="test"
)

split_order = [
    "train",
    "validation",
    "test"
]

initial_split_counts = (
    df_split["data_split"]
    .value_counts()
    .reindex(split_order, fill_value=0)
)

display(
    initial_split_counts
    .rename_axis("data_split")
    .reset_index(name="appointments")
)

assert (initial_split_counts > 0).all()


# 6. Verify outcome maturity.

print("\n5. OUTCOME MATURITY CHECK")

df_split["eligible_for_split"] = True
df_split["exclusion_reason"] = ""

first_validation_cutoff = df_split.loc[
    df_split["data_split"] == "validation",
    "prediction_cutoff_datetime"
].min()

first_test_cutoff = df_split.loc[
    df_split["data_split"] == "test",
    "prediction_cutoff_datetime"
].min()



train_not_mature = (
    (df_split["data_split"] == "train")
    & (
        df_split["appointment_datetime"]
        >= first_validation_cutoff
    )
)

validation_not_mature = (
    (df_split["data_split"] == "validation")
    & (
        df_split["appointment_datetime"]
        >= first_test_cutoff
    )
)

df_split.loc[
    train_not_mature,
    "eligible_for_split"
] = False

df_split.loc[
    train_not_mature,
    "exclusion_reason"
] = "Outcome not mature before validation prediction"

df_split.loc[
    validation_not_mature,
    "eligible_for_split"
] = False

df_split.loc[
    validation_not_mature,
    "exclusion_reason"
] = "Outcome not mature before test prediction"

print(
    "Training boundary exclusions:",
    int(train_not_mature.sum())
)

print(
    "Validation boundary exclusions:",
    int(validation_not_mature.sum())
)

print(
    "Test boundary exclusions: 0"
)



# 7. Build the usable split datasets.

print("\n6. CREATE USABLE SPLITS")

df_train = df_split.loc[
    (df_split["data_split"] == "train")
    & df_split["eligible_for_split"]
].copy()

df_val = df_split.loc[
    (df_split["data_split"] == "validation")
    & df_split["eligible_for_split"]
].copy()

df_test = df_split.loc[
    (df_split["data_split"] == "test")
    & df_split["eligible_for_split"]
].copy()

assert all(
    len(part) > 0
    for part in [df_train, df_val, df_test]
)

# Check that all classes occur in each split.

for name, part in [
    ("train", df_train),
    ("validation", df_val),
    ("test", df_test)
]:

    assert part["dna_target"].nunique() == 2, (
        f"{name} does not contain both outcomes."
    )

    print(
        f"{name}: {len(part):,} appointments, "
        f"DNA rate {part['dna_target'].mean() * 100:.2f}%"
    )


# 8. Separate predictors and target.


print("\n7. CREATE PREDICTOR AND TARGET SETS")

X_train = df_train[
    compact_candidate_features
].copy()

X_val = df_val[
    compact_candidate_features
].copy()

X_test = df_test[
    compact_candidate_features
].copy()

y_train = df_train["dna_target"].copy()
y_val = df_val["dna_target"].copy()
y_test = df_test["dna_target"].copy()

assert list(X_train.columns) == list(X_val.columns)
assert list(X_train.columns) == list(X_test.columns)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)


# 9. Check for appointment overlap.
# Repeated patients are permitted across periods,
# but patient_id is never a predictor.

print("\n8. IDENTIFIER LEAKAGE CHECKS")

train_ids = set(df_train["appointment_id"])
val_ids = set(df_val["appointment_id"])
test_ids = set(df_test["appointment_id"])

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

train_patients = set(df_train["patient_id"])
val_patients = set(df_val["patient_id"])
test_patients = set(df_test["patient_id"])

patient_overlap = pd.DataFrame([
    {
        "comparison": "Train vs validation",
        "shared_patients": len(
            train_patients & val_patients
        )
    },
    {
        "comparison": "Train vs test",
        "shared_patients": len(
            train_patients & test_patients
        )
    },
    {
        "comparison": "Validation vs test",
        "shared_patients": len(
            val_patients & test_patients
        )
    }
])

display(patient_overlap)

print("Appointment overlap: NONE")

print(
    "Repeated patients are documented. "
    "A patient-disjoint sensitivity analysis "
    "may be useful during evaluation."
)


# 10. Verify chronological boundaries.

print("\n9. VERIFY TEMPORAL ORDER")

assert (
    df_train["appointment_datetime"].max()
    < df_val["appointment_datetime"].min()
)

assert (
    df_val["appointment_datetime"].max()
    < df_test["appointment_datetime"].min()
)

assert (
    df_train["appointment_datetime"].max()
    < first_validation_cutoff
)

assert (
    df_val["appointment_datetime"].max()
    < first_test_cutoff
)

print("Appointment-date ordering: PASS")
print("Outcome-maturity ordering: PASS")


# 11. Create a split summary.

print("\n10. SPLIT SUMMARY")

split_summary_rows = []

for name, part in [
    ("train", df_train),
    ("validation", df_val),
    ("test", df_test)
]:

    split_summary_rows.append({
        "split": name,
        "appointments": len(part),
        "dna_cases": int(part["dna_target"].sum()),
        "dna_rate_percent": round(
            part["dna_target"].mean() * 100,
            2
        ),
        "first_appointment": str(
            part["appointment_datetime"].min()
        ),
        "last_appointment": str(
            part["appointment_datetime"].max()
        ),
        "first_prediction_cutoff": str(
            part["prediction_cutoff_datetime"].min()
        ),
        "last_prediction_cutoff": str(
            part["prediction_cutoff_datetime"].max()
        )
    })

split_summary = pd.DataFrame(
    split_summary_rows
)

display(split_summary)


# 12. Keep fairness variables for later auditing.

print("\n11. PRESERVE FAIRNESS METADATA")

fairness_columns = [
    "appointment_id",
    "patient_id",
    "dna_target",
    "age",
    "sex",
    "ethnic_group",
    "imd_decile",
    "data_split",
    "eligible_for_split"
]

fairness_split_audit = df_split[
    fairness_columns
].copy()

assert len(fairness_split_audit) == len(df_split)

print("Fairness metadata preserved.")


# 13. Save split manifests and reports.


print("\n12. SAVE CHRONOLOGICAL SPLIT REPORTS")

split_manifest_columns = [
    "appointment_id",
    "patient_id",
    "booked_datetime",
    "prediction_cutoff_datetime",
    "appointment_datetime",
    "dna_target",
    "data_split",
    "eligible_for_split",
    "exclusion_reason"
]

split_manifest = df_split[
    split_manifest_columns
].copy()

boundary_exclusions = split_manifest.loc[
    ~split_manifest["eligible_for_split"]
].copy()

cell06_reports = {
    "cell06_split_summary.csv":
        split_summary,

    "cell06_split_manifest.csv":
        split_manifest,

    "cell06_boundary_exclusions.csv":
        boundary_exclusions,

    "cell06_patient_overlap.csv":
        patient_overlap
}

for filename, table in cell06_reports.items():

    output_path = FEATURE_REPORT_DIR / filename

    table.to_csv(
        output_path,
        index=False
    )

    print(f"Saved: {output_path}")


# 14. Final integrity checks.

print("\n13. FINAL CELL 6 VALIDATION")

usable_count = (
    len(df_train) + len(df_val) + len(df_test)
)

excluded_count = int(
    (~df_split["eligible_for_split"]).sum()
)

assert usable_count + excluded_count == len(df_split)

assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)
assert len(X_test) == len(y_test)

assert not (
    {
        "dna_target",
        "attendance_status",
        "clinic_checkin_time",
        "patient_id",
        "appointment_id",
        "previous_dnas_12m",
        "rescheduled_count",
        "sms_reminder_sent"
    }
    & set(X_train.columns)
)

print(f"Original appointments: {len(df_split):,}")
print(f"Usable appointments: {usable_count:,}")
print(f"Boundary exclusions: {excluded_count:,}")
print(f"Predictor columns: {X_train.shape[1]}")
print(f"Reports saved: {len(cell06_reports)}")

print(
    "\nNo preprocessing fitted and "
    "no models trained."
)

print("COMPLETED SUCCESSFULLY")



CELL 6: CHRONOLOGICAL DATA SPLITTING
Input appointments: 36,264
Candidate predictors: 15

1. VALIDATE CHRONOLOGY
Chronology: PASS

2. SORT APPOINTMENTS
First appointment: 2025-04-01 08:30:00
Last appointment: 2026-03-31 16:30:00

3. DEFINE SPLIT BOUNDARIES
Validation period begins: 2025-12-15 00:00:00
Test period begins: 2026-02-05 00:00:00

4. ASSIGN SPLITS


,data_split,appointments
0,train,25297
1,validation,5479
2,test,5488



5. OUTCOME MATURITY CHECK
Training boundary exclusions: 685
Validation boundary exclusions: 706
Test boundary exclusions: 0

6. CREATE USABLE SPLITS
train: 24,612 appointments, DNA rate 9.06%
validation: 4,773 appointments, DNA rate 9.11%
test: 5,488 appointments, DNA rate 9.55%

7. CREATE PREDICTOR AND TARGET SETS
X_train: (24612, 15)
X_val: (4773, 15)
X_test: (5488, 15)

8. IDENTIFIER LEAKAGE CHECKS


,comparison,shared_patients
0,Train vs validation,2945
1,Train vs test,3320
2,Validation vs test,1218


Appointment overlap: NONE
Repeated patients are documented. A patient-disjoint sensitivity analysis may be useful during evaluation.

9. VERIFY TEMPORAL ORDER
Appointment-date ordering: PASS
Outcome-maturity ordering: PASS

10. SPLIT SUMMARY


,split,appointments,dna_cases,dna_rate_percent,first_appointment,last_appointment,first_prediction_cutoff,last_prediction_cutoff
0,train,24612,2230,9.06,2025-04-01 08:30:00,2025-12-05 16:30:00,2025-03-25 08:30:00,2025-12-03 11:00:00
1,validation,4773,435,9.11,2025-12-15 08:30:00,2026-01-28 16:30:00,2025-12-08 08:30:00,2026-01-27 14:42:00
2,test,5488,524,9.55,2026-02-05 08:30:00,2026-03-31 16:30:00,2026-01-29 08:30:00,2026-03-30 10:24:00



11. PRESERVE FAIRNESS METADATA
Fairness metadata preserved.

12. SAVE CHRONOLOGICAL SPLIT REPORTS
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell06_split_summary.csv
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell06_split_manifest.csv
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell06_boundary_exclusions.csv
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell06_patient_overlap.csv

13. FINAL CELL 6 VALIDATION
Original appointments: 36,264
Usable appointments: 34,873
Boundary exclusions: 1,391
Predictor columns: 15
Reports saved: 4

No preprocessing fitted and no models trained.
COMPLETED SUCCESSFULLY


In [16]:

# FINAL FEATURE ENGINEERING HANDOVER

#
# Purpose:
# Save the engineered dataset, chronological split
# assignments, selected features and audit summary
# for use in 05_model_development.ipynb.
#
# No rows are removed, no preprocessing is fitted,
# and no machine learning models are trained.

from pathlib import Path
import json
import pandas as pd

print("CELL 7: FINAL FEATURE ENGINEERING HANDOVER")


# 1. Validate outputs from previous cells.

print("\n1. VERIFY FEATURE ENGINEERING OUTPUTS")

required_objects = [
    "df_split",
    "df_train",
    "df_val",
    "df_test",
    "compact_candidate_features",
    "PROJECT_ROOT"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    f"Run Cells 1-6 first. Missing: {missing_objects}"
)

assert len(df_split) == 36264
assert df_split["appointment_id"].is_unique

assert len(compact_candidate_features) == 15

assert all(
    feature in df_split.columns
    for feature in compact_candidate_features
)

print("Previous cells validated: PASS")


# 2. Define output directories.

print("\n2. CREATE HANDOVER DIRECTORIES")

handover_dir = (
    Path(PROJECT_ROOT) / "data" / "processed"
)

report_dir = (
    Path(PROJECT_ROOT) / "reports" / "feature_engineering"
)

handover_dir.mkdir(
    parents=True,
    exist_ok=True
)

report_dir.mkdir(
    parents=True,
    exist_ok=True
)

print("Data directory:", handover_dir)
print("Report directory:", report_dir)


# 3. Prepare the complete engineered dataset.
#
# This includes the original cohort, feature
# columns, split assignments and exclusions.


print("\n3. PREPARE ENGINEERED DATASET")

df_handover = df_split.copy(deep=True)

assert len(df_handover) == 36264

assert {
    "appointment_id",
    "patient_id",
    "dna_target",
    "data_split",
    "eligible_for_split",
    "exclusion_reason"
}.issubset(df_handover.columns)

assert (
    df_handover["data_split"]
    .isin(["train", "validation", "test"])
).all()

assert df_handover["eligible_for_split"].notna().all()

print("Rows:", len(df_handover))
print("Columns:", df_handover.shape[1])


# 4. Validate usable split sizes.

print("\n4. VERIFY CHRONOLOGICAL SPLITS")

expected_counts = {
    "train": 24612,
    "validation": 4773,
    "test": 5488
}

actual_counts = {}

for split_name in expected_counts:

    count = int(
        (
            (df_handover["data_split"] == split_name)
            &
            df_handover["eligible_for_split"]
        ).sum()
    )

    actual_counts[split_name] = count

    assert count == expected_counts[split_name], (
        f"Unexpected {split_name} count: {count}"
    )

    print(f"{split_name}: {count:,}")

usable_count = sum(actual_counts.values())

excluded_count = int(
    (~df_handover["eligible_for_split"]).sum()
)

assert usable_count == 34873
assert excluded_count == 1391
assert usable_count + excluded_count == 36264

print("Usable appointments:", usable_count)
print("Boundary exclusions:", excluded_count)


# 5. Define the approved handover feature list.


print("\n5. VERIFY CANDIDATE FEATURES")

selected_features = list(
    compact_candidate_features
)

forbidden_features = {
    "dna_target",
    "attendance_status",
    "clinic_checkin_time",
    "appointment_id",
    "patient_id",
    "booked_datetime",
    "appointment_datetime",
    "prediction_cutoff_datetime",
    "previous_dnas_12m",
    "rescheduled_count",
    "sms_reminder_sent",
    "sex",
    "ethnic_group",
    "imd_decile",
    "data_split",
    "eligible_for_split",
    "exclusion_reason"
}

assert not (
    set(selected_features) & forbidden_features
)

assert len(selected_features) == len(
    set(selected_features)
)

for feature in selected_features:
    print(" -", feature)

print("Predictor leakage exclusion check: PASS")


# 6. Save engineered dataset.

print("\n6. SAVE ENGINEERED DATASET")

dataset_path = (
    handover_dir /
    "outpatient_dna_feature_engineered.csv"
)

df_handover.to_csv(
    dataset_path,
    index=False
)

print("Saved:", dataset_path)


# 7. Save the 15 candidate feature names.

print("\n7. SAVE SELECTED FEATURE LIST")

features_path = (
    handover_dir /
    "outpatient_dna_selected_features.json"
)

with open(
    features_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        {
            "feature_count": len(selected_features),
            "features": selected_features,
            "status": "provisional",
            "note": (
                "Verify feature availability at "
                "prediction cutoff before deployment."
            )
        },
        file,
        indent=4
    )

print("Saved:", features_path)


# 8. Save split manifest separately.
# This allows Notebook 05 to reconstruct
# the exact same chronological partitions.

print("\n8. SAVE SPLIT MANIFEST")

manifest_columns = [
    "appointment_id",
    "patient_id",
    "prediction_cutoff_datetime",
    "appointment_datetime",
    "dna_target",
    "data_split",
    "eligible_for_split",
    "exclusion_reason"
]

split_manifest_handover = df_handover[
    manifest_columns
].copy()

manifest_path = (
    handover_dir /
    "outpatient_dna_split_manifest.csv"
)

split_manifest_handover.to_csv(
    manifest_path,
    index=False
)

print("Saved:", manifest_path)


# 9. Save a summary of usable data.

print("\n9. SAVE HANDOVER SUMMARY")

summary_rows = []

for split_name in [
    "train",
    "validation",
    "test"
]:

    part = df_handover.loc[
        (
            df_handover["data_split"] == split_name
        )
        &
        df_handover["eligible_for_split"]
    ]

    summary_rows.append({
        "split": split_name,
        "appointments": len(part),
        "dna_cases": int(part["dna_target"].sum()),
        "dna_rate_percent": round(
            part["dna_target"].mean() * 100,
            2
        ),
        "candidate_predictors": len(selected_features)
    })

handover_summary = pd.DataFrame(
    summary_rows
)

display(handover_summary)

summary_path = (
    report_dir /
    "cell07_final_handover_summary.csv"
)

handover_summary.to_csv(
    summary_path,
    index=False
)

print("Saved:", summary_path)


# 10. Verify files can be read independently.


print("\n10. VERIFY SAVED HANDOVER FILES")

reloaded_data = pd.read_csv(
    dataset_path,
    low_memory=False
)

reloaded_manifest = pd.read_csv(
    manifest_path,
    low_memory=False
)

with open(
    features_path,
    "r",
    encoding="utf-8"
) as file:

    reloaded_features = json.load(file)

assert len(reloaded_data) == 36264
assert len(reloaded_manifest) == 36264

assert reloaded_data["appointment_id"].is_unique

assert (
    reloaded_features["features"]
    == selected_features
)

assert all(
    feature in reloaded_data.columns
    for feature in selected_features
)

print("Engineered dataset reload: PASS")
print("Split manifest reload: PASS")
print("Feature list reload: PASS")


# 11. Final handover validation.

print("\n11. FINAL VALIDATION")

assert int(
    df_handover["dna_target"].sum()
) == 3306

assert len(df_train) == expected_counts["train"]
assert len(df_val) == expected_counts["validation"]
assert len(df_test) == expected_counts["test"]

assert sum(
    handover_summary["dna_cases"]
) == 3189

print("Original appointments: 36,264")
print("Usable appointments: 34,873")
print("Boundary exclusions: 1,391")
print("Selected candidate predictors: 15")
print("Handover files: 4")

print(
    "\nNo model trained and no preprocessing fitted."
)

print("COMPLETED SUCCESSFULLY")
print("FEATURE ENGINEERING HANDOVER COMPLETE")
print("Ready for 05_model_development.ipynb")


CELL 7: FINAL FEATURE ENGINEERING HANDOVER

1. VERIFY FEATURE ENGINEERING OUTPUTS
Previous cells validated: PASS

2. CREATE HANDOVER DIRECTORIES
Data directory: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\data\processed
Report directory: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering

3. PREPARE ENGINEERED DATASET
Rows: 36264
Columns: 53

4. VERIFY CHRONOLOGICAL SPLITS
train: 24,612
validation: 4,773
test: 5,488
Usable appointments: 34873
Boundary exclusions: 1391

5. VERIFY CANDIDATE FEATURES
 - age
 - age_missing_flag
 - distance_km
 - distance_missing_flag
 - long_term_conditions
 - interpreter_required
 - specialty
 - appointment_type
 - consultation_mode
 - referral_source
 - booking_lead_days_exact
 - is_short_notice
 - appointment_day_of_week
 - appointment_month
 - appointment_hour
Predictor leakage exclusion check: PASS

6. SAVE ENGINEERED DATASET
Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\d

,split,appointments,dna_cases,dna_rate_percent,candidate_predictors
0,train,24612,2230,9.06,15
1,validation,4773,435,9.11,15
2,test,5488,524,9.55,15


Saved: c:\Users\User.DESKTOP-775\Documents\nhs-dorset-dna-prediction\reports\feature_engineering\cell07_final_handover_summary.csv

10. VERIFY SAVED HANDOVER FILES
Engineered dataset reload: PASS
Split manifest reload: PASS
Feature list reload: PASS

11. FINAL VALIDATION
Original appointments: 36,264
Usable appointments: 34,873
Boundary exclusions: 1,391
Selected candidate predictors: 15
Handover files: 4

No model trained and no preprocessing fitted.
COMPLETED SUCCESSFULLY
FEATURE ENGINEERING HANDOVER COMPLETE
Ready for 05_model_development.ipynb
